# 測試&單股評估&debug cell 0-7  ;全股評估 cell 0,1,8

In [28]:
#------------------------------------------------------------------
#Cell 0 — 核心評價函式
#------------------------------------------------------------------
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import date
from typing import Any, Protocol, Iterable
import logging
import math

import pandas as pd
import yaml

LOGGER = logging.getLogger(__name__)


# =========================
# Config
# =========================
def load_config(path: str | Path | None = None):

    config_path = (
        Path(path)
        if path
        else Path("config.yaml")
    )


    with config_path.open(
        encoding="utf-8"
    ) as file:

        config = yaml.safe_load(file)


    # 加入設定檔名稱
    config["_config_name"] = (
        config_path.stem
    )


    return config

# =========================
# Shared helpers
# =========================
def safe_divide(numerator: float | None, denominator: float | None) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    result = numerator / denominator
    return result if math.isfinite(result) else None


def percentile_rank(value: float | None, values: Iterable[float]) -> float | None:
    cleaned = sorted(item for item in values if item is not None and math.isfinite(item))
    if value is None or not cleaned:
        return None
    return 100 * sum(item <= value for item in cleaned) / len(cleaned)


# =========================
# Data layer
# =========================
class DataProvider(Protocol):
    def get_snapshot(self, ticker: str) -> "FinancialSnapshot":
        ...

    def get_price_history(self, ticker: str, period: str = "5y") -> pd.DataFrame:
        ...

    def get_historical_fundamentals(self, ticker: str) -> pd.DataFrame:
        ...

    def get_chip_data(self, ticker: str) -> pd.DataFrame:
        ...

    def get_institutional_data(self, ticker: str) -> pd.DataFrame:
        """
        Expected columns:
        - report_date
        - institutional_ownership_pct
        - holders_count
        """
        ...

    def get_insider_data(self, ticker: str) -> pd.DataFrame:
        """
        Expected columns:
        - date
        - transaction_type   # buy / sell
        - shares
        - value
        """
        ...


@dataclass
class FinancialSnapshot:
    ticker: str
    source: str
    as_of: date
    asset_type: str | None = None
    current_price: float | None = None
    week_52_high: float | None = None
    week_52_low: float | None = None
    market_cap: float | None = None
    enterprise_value: float | None = None
    revenue: float | None = None
    gross_profit: float | None = None
    operating_income: float | None = None
    net_income: float | None = None
    eps: float | None = None
    ttm_eps: float | None = None
    eps_ttm_nowcast: float | None = None
    pe_ttm_nowcast: float | None = None
    forward_eps: float | None = None
    book_value: float | None = None
    book_value_per_share: float | None = None
    free_cash_flow: float | None = None
    operating_cash_flow: float | None = None
    capex: float | None = None
    total_debt: float | None = None
    cash: float | None = None
    shares_outstanding: float | None = None
    average_shares_12m: float | None = None
    total_equity: float | None = None
    current_assets: float | None = None
    current_liabilities: float | None = None
    inventory: float | None = None
    interest_expense: float | None = None
    financial_history: dict[str, list[float]] = field(default_factory=dict)


@dataclass
class Classification:
    market: str
    sector: str

class YahooFinanceProvider:
    source_name = "Yahoo Finance (yfinance)"

    def _ticker(self, ticker: str) -> Any:
        try:
            import yfinance as yf
        except ImportError as exc:
            raise ImportError("Please install yfinance: pip install yfinance") from exc
        return yf.Ticker(ticker)

    @staticmethod
    def _value(frame: pd.DataFrame, label: str) -> float | None:
        if frame is None or frame.empty or label not in frame.index:
            return None
        values = frame.loc[label].dropna()
        return float(values.iloc[0]) if not values.empty else None

    @staticmethod
    def _row_values(frame: pd.DataFrame, label: str) -> list[float]:
        if frame is None or frame.empty or label not in frame.index:
            return []
        return [float(item) for item in frame.loc[label].dropna().tolist()]

    @staticmethod
    def _cell(frame: pd.DataFrame, label: str, column: object) -> float | None:
        if frame is None or frame.empty or label not in frame.index:
            return None
        value = frame.loc[label, column]
        return float(value) if pd.notna(value) else None

    @classmethod
    def _latest_ttm_eps(cls, quarterly_income: pd.DataFrame) -> float | None:
        if quarterly_income is None or quarterly_income.empty:
            return None

        quarterly_dates = sorted(quarterly_income.columns, reverse=True)
        values: list[float] = []
        for report_date in quarterly_dates:
            eps = cls._cell(quarterly_income, "Diluted EPS", report_date)
            if eps is None:
                eps = cls._cell(quarterly_income, "Basic EPS", report_date)
            if eps is None:
                net_income = cls._cell(quarterly_income, "Net Income", report_date)
                shares = cls._cell(quarterly_income, "Basic Average Shares", report_date)
                eps = safe_divide(net_income, shares)
            if eps is None:
                return None
            values.append(eps)
            if len(values) == 4:
                return float(sum(values))
        return None

    def _history(
        self,
        income: pd.DataFrame,
        cashflow: pd.DataFrame,
        quarterly_income: pd.DataFrame,
    ) -> dict[str, list[float]]:
        return {
            "revenue": self._row_values(income, "Total Revenue"),
            "net_income": self._row_values(income, "Net Income"),
            "eps": self._row_values(income, "Diluted EPS") or self._row_values(income, "Basic EPS"),
            "free_cash_flow": self._row_values(cashflow, "Free Cash Flow"),
            "quarterly_revenue": self._row_values(quarterly_income, "Total Revenue"),
            "quarterly_eps": self._row_values(quarterly_income, "Diluted EPS")
            or self._row_values(quarterly_income, "Basic EPS"),
        }

    def get_snapshot(self, ticker: str) -> FinancialSnapshot:
        instrument = self._ticker(ticker)
        info = instrument.info
        income = instrument.financials
        balance = instrument.balance_sheet
        cashflow = instrument.cashflow
        quarterly_income = instrument.quarterly_income_stmt
        ttm_eps = self._latest_ttm_eps(quarterly_income)

        return FinancialSnapshot(
            ticker=ticker,
            source=self.source_name,
            as_of=date.today(),
            asset_type=info.get("quoteType"),
            current_price=info.get("currentPrice") or info.get("regularMarketPrice"),
            week_52_high=info.get("fiftyTwoWeekHigh"),
            week_52_low=info.get("fiftyTwoWeekLow"),
            market_cap=info.get("marketCap"),
            enterprise_value=info.get("enterpriseValue"),
            revenue=self._value(income, "Total Revenue"),
            gross_profit=self._value(income, "Gross Profit"),
            operating_income=self._value(income, "Operating Income"),
            net_income=self._value(income, "Net Income"),
            eps=ttm_eps if ttm_eps is not None else info.get("trailingEps"),
            ttm_eps=ttm_eps,
            forward_eps=info.get("forwardEps"),
            book_value=info.get("bookValue"),
            book_value_per_share=info.get("bookValue"),
            free_cash_flow=info.get("freeCashflow") or self._value(cashflow, "Free Cash Flow"),
            operating_cash_flow=self._value(cashflow, "Operating Cash Flow"),
            capex=self._value(cashflow, "Capital Expenditure"),
            total_debt=info.get("totalDebt") or self._value(balance, "Total Debt"),
            cash=info.get("totalCash"),
            shares_outstanding=info.get("sharesOutstanding"),
            total_equity=self._value(balance, "Stockholders Equity"),
            current_assets=self._value(balance, "Current Assets"),
            current_liabilities=self._value(balance, "Current Liabilities"),
            inventory=self._value(balance, "Inventory"),
            interest_expense=self._value(income, "Interest Expense"),
            financial_history=self._history(income, cashflow, quarterly_income),
        )

    def get_price_history(self, ticker: str, period: str = "5y") -> pd.DataFrame:
        return self._ticker(ticker).history(period=period, auto_adjust=False)

    def get_historical_fundamentals(self, ticker: str) -> pd.DataFrame:
        """
        Return dated TTM EPS and BVPS observations.
        Priority:
        1. Quarterly statements -> TTM EPS + quarterly BVPS
        2. Fallback to annual statements if quarterly data is insufficient
        """
        instrument = self._ticker(ticker)
        q_income = getattr(instrument, "quarterly_income_stmt", None)
        q_balance = getattr(instrument, "quarterly_balance_sheet", None)

        rows: list[dict[str, float | pd.Timestamp | None]] = []

        # Quarterly path: build TTM EPS
        if q_income is not None and not q_income.empty:
            quarter_dates = list(q_income.columns)
            quarter_eps_map: dict[pd.Timestamp, float | None] = {}

            for col in quarter_dates:
                col_ts = pd.Timestamp(col)

                eps = self._cell(q_income, "Diluted EPS", col)
                if eps is None:
                    eps = self._cell(q_income, "Basic EPS", col)

                if eps is None:
                    net_income = self._cell(q_income, "Net Income", col)
                    shares = self._cell(q_income, "Basic Average Shares", col)
                    if net_income is not None and shares not in (None, 0):
                        eps = net_income / shares

                quarter_eps_map[col_ts] = eps

            sorted_dates = sorted(quarter_eps_map.keys())

            for i in range(3, len(sorted_dates)):
                report_date = sorted_dates[i]
                last_4 = sorted_dates[i - 3:i + 1]
                eps_values = [quarter_eps_map[d] for d in last_4]

                if any(v is None for v in eps_values):
                    continue

                ttm_eps = float(sum(eps_values))  # type: ignore[arg-type]

                bvps = None
                if q_balance is not None and not q_balance.empty and report_date in q_balance.columns:
                    equity = self._cell(q_balance, "Stockholders Equity", report_date)
                    shares = self._cell(q_balance, "Ordinary Shares Number", report_date)

                    if shares in (None, 0):
                        shares = self._cell(q_balance, "Share Issued", report_date)

                    if equity is not None and shares not in (None, 0):
                        bvps = equity / shares

                rows.append({
                    "report_date": report_date,
                    "ttm_eps": ttm_eps,
                    "bvps": bvps,
                })

        quarterly_dates = {pd.Timestamp(row["report_date"]) for row in rows}

        # Add annual history to extend the valuation lookback beyond Yahoo's limited quarterly history.
        income = getattr(instrument, "income_stmt", None)
        balance = getattr(instrument, "balance_sheet", None)

        if income is None or income.empty:
            return pd.DataFrame(rows).sort_values("report_date").reset_index(drop=True) if rows else pd.DataFrame(columns=["report_date", "ttm_eps", "bvps"])

        fallback_rows: list[dict[str, float | pd.Timestamp | None]] = []

        for col in income.columns:
            col_ts = pd.Timestamp(col)
            if col_ts in quarterly_dates:
                continue

            eps = self._cell(income, "Basic EPS", col)
            if eps is None:
                net_income = self._cell(income, "Net Income", col)
                shares = self._cell(income, "Basic Average Shares", col)
                if net_income is not None and shares not in (None, 0):
                    eps = net_income / shares

            bvps = None
            if balance is not None and not balance.empty and col in balance.columns:
                equity = self._cell(balance, "Stockholders Equity", col)
                shares = self._cell(balance, "Ordinary Shares Number", col)

                if shares in (None, 0):
                    shares = self._cell(balance, "Share Issued", col)

                if equity is not None and shares not in (None, 0):
                    bvps = equity / shares

            fallback_rows.append({
                "report_date": col_ts,
                "ttm_eps": eps,
                "bvps": bvps,
            })

        return pd.DataFrame(rows + fallback_rows).sort_values("report_date").reset_index(drop=True)

    def get_chip_data(self, ticker: str) -> pd.DataFrame:
        LOGGER.info("Chip data unavailable for %s from %s", ticker, self.source_name)
        return pd.DataFrame(columns=["date", "institutional_net_buy", "margin_balance"])

    def get_institutional_data(self, ticker: str) -> pd.DataFrame:
        LOGGER.info("Institutional data unavailable for %s from %s", ticker, self.source_name)
        return pd.DataFrame(columns=["report_date", "institutional_ownership_pct", "holders_count"])

    def get_insider_data(self, ticker: str) -> pd.DataFrame:
        LOGGER.info("Insider data unavailable for %s from %s", ticker, self.source_name)
        return pd.DataFrame(columns=["date", "transaction_type", "shares", "value"])

class FinMindFundamentalProvider:
    """
    第一階段：台股歷史基本面 Provider。

    用途：
    - 從 FinMind 抓取損益表、資產負債表
    - 建立歷史季度 EPS、TTM EPS、BVPS
    - 回傳格式可直接交給 ValuationEngine：
        report_date | ttm_eps | bvps

    注意：
    台灣財報中的 EPS 常為年初至今累積值：
    Q1 = Q1
    Q2 = 上半年累積
    Q3 = 前三季累積
    Q4 = 全年累積

    因此會先轉換成單季 EPS，再計算 TTM EPS。
    """

    BASE_URL = "https://api.finmindtrade.com/api/v4/data"

    def __init__(self, config: dict[str, Any]) -> None:
        settings = config.get("data_sources", {}).get("finmind", {})

        self.enabled = settings.get("enabled", False)
        self.token = settings.get("token")

        # 台灣上市櫃公司通常面額為 10 元。
        # 如遇特殊面額個股，未來可在 config 加個別 override。
        self.default_par_value = settings.get("par_value_default", 10.0)

        # 個別股票面額 override，例如：
        # ticker_par_values:
        #   "XXXX.TW": 10.0
        self.ticker_par_values = settings.get("ticker_par_values", {})
        
        # 避免同一輪分析重複呼叫 FinMind API
        self._income_cache: dict[str, pd.DataFrame] = {}
        self._balance_cache: dict[str, pd.DataFrame] = {}
        self._cashflow_cache: dict[str, pd.DataFrame] = {}
        self._month_revenue_cache: dict[str, pd.DataFrame] = {}

    @staticmethod
    def is_taiwan_ticker(ticker: str) -> bool:
        return ticker.endswith((".TW", ".TWO"))

    @staticmethod
    def ticker_code(ticker: str) -> str:
        return ticker.split(".")[0]

    def _request(
        self,
        dataset: str,
        stock_id: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """使用 FinMind API v4 抓取資料。"""

        if not self.enabled:
            LOGGER.info("FinMind disabled.")
            return pd.DataFrame()

        if not self.token:
            LOGGER.warning("FinMind enabled but token is missing.")
            return pd.DataFrame()

        try:
            import requests
        except ImportError as exc:
            raise ImportError(
                "Please install requests: pip install requests"
            ) from exc

        params = {
            "dataset": dataset,
            "data_id": stock_id,
            "start_date": start_date,
            "token": self.token,
        }

        try:
            response = requests.get(
                self.BASE_URL,
                params=params,
                timeout=30,
            )
            response.raise_for_status()

            payload = response.json()
            data = payload.get("data", [])

            if not data:
                LOGGER.warning(
                    "FinMind returned no data: dataset=%s, stock=%s",
                    dataset,
                    stock_id,
                )
                return pd.DataFrame()

            return pd.DataFrame(data)

        except Exception as exc:
            LOGGER.warning(
                "FinMind request failed: dataset=%s, stock=%s, error=%s",
                dataset,
                stock_id,
                exc,
            )
            return pd.DataFrame()

    def get_financial_statements(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """取得台股損益表資料，並使用記憶體快取。"""

        if not self.is_taiwan_ticker(ticker):
            return pd.DataFrame()

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._income_cache:
            self._income_cache[cache_key] = self._request(
                dataset="TaiwanStockFinancialStatements",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._income_cache[cache_key].copy()    
    
    def get_balance_sheet(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """取得台股資產負債表資料，並使用記憶體快取。"""

        if not self.is_taiwan_ticker(ticker):
            return pd.DataFrame()

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._balance_cache:
            self._balance_cache[cache_key] = self._request(
                dataset="TaiwanStockBalanceSheet",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._balance_cache[cache_key].copy()
    
    def get_cash_flow_statement(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """
        取得台股現金流量表資料，並使用記憶體快取。

        FinMind Dataset：
        TaiwanStockCashFlowsStatement
        """
        if not self.is_taiwan_ticker(ticker):
            return pd.DataFrame()

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._cashflow_cache:
            self._cashflow_cache[cache_key] = self._request(
                dataset="TaiwanStockCashFlowsStatement",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._cashflow_cache[cache_key].copy()
    
    def get_month_revenue(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """
        取得台股月營收資料。

        FinMind Dataset:
        TaiwanStockMonthRevenue

        常見欄位：
        - date
        - stock_id
        - country
        - revenue
        - revenue_month
        - revenue_year
        """
        if not self.is_taiwan_ticker(ticker):
            return pd.DataFrame()

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._month_revenue_cache:
            self._month_revenue_cache[cache_key] = self._request(
                dataset="TaiwanStockMonthRevenue",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._month_revenue_cache[cache_key].copy()
    
    def get_eps_ttm_nowcast_inputs(  
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, float | None]:
        """
        建立 EPS TTM Nowcast 所需資料。

        公式：
        EPS TTM Nowcast
        =
        最近 12 個已公告月營收
        × 最近三季營收加權平均淨利率
        ÷ 最近四季平均股數

        注意：
        - 月營收歸屬月份以 revenue_year / revenue_month 為準。
        - date 通常是公告月份第一天，不能直接視為營收所屬月份。
        """

        result = {
            "ltm_month_revenue": None,
            "weighted_net_margin_3q": None,
            "average_shares_12m": None,
            "eps_ttm_nowcast": None,
        }

        if not self.is_taiwan_ticker(ticker):
            return result

        # ---------------------------------------------------------
        # 1. 最近 12 個已公告月營收
        # ---------------------------------------------------------
        month_revenue_df = self.get_month_revenue(
            ticker,
            start_date,
        )

        required_month_cols = {
            "revenue",
            "revenue_year",
            "revenue_month",
        }

        if (
            month_revenue_df is None
            or month_revenue_df.empty
            or not required_month_cols.issubset(month_revenue_df.columns)
        ):
            return result

        month_revenue_df = month_revenue_df.copy()

        month_revenue_df["revenue"] = pd.to_numeric(
            month_revenue_df["revenue"],
            errors="coerce",
        )

        month_revenue_df["revenue_year"] = pd.to_numeric(
            month_revenue_df["revenue_year"],
            errors="coerce",
        )

        month_revenue_df["revenue_month"] = pd.to_numeric(
            month_revenue_df["revenue_month"],
            errors="coerce",
        )

        month_revenue_df = month_revenue_df.dropna(
            subset=[
                "revenue",
                "revenue_year",
                "revenue_month",
            ]
        )

        # 用真正的營收歸屬年月建立日期欄位
        month_revenue_df["revenue_period"] = pd.to_datetime(
            {
                "year": month_revenue_df["revenue_year"].astype(int),
                "month": month_revenue_df["revenue_month"].astype(int),
                "day": 1,
            }
        )

        month_revenue_df = (
            month_revenue_df
            .sort_values("revenue_period")
            .drop_duplicates(
                subset=["revenue_period"],
                keep="last",
            )
            .reset_index(drop=True)
        )

        latest_12_months = month_revenue_df.tail(12)

        # 必須有完整 12 個月才建立 Nowcast
        if len(latest_12_months) != 12:
            LOGGER.warning(
                "FinMind month revenue has fewer than 12 observations for %s",
                ticker,
            )
            return result

        ltm_month_revenue = float(
            latest_12_months["revenue"].sum()
        )

        result["ltm_month_revenue"] = ltm_month_revenue

        # ---------------------------------------------------------
        # 2. 最近三季營收加權平均淨利率
        # ---------------------------------------------------------
        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        required_income_cols = {
            "date",
            "type",
            "value",
        }

        if (
            income_df is None
            or income_df.empty
            or not required_income_cols.issubset(income_df.columns)
        ):
            return result

        income_df = income_df.copy()

        income_df["date"] = pd.to_datetime(
            income_df["date"],
            errors="coerce",
        )

        income_df["value"] = pd.to_numeric(
            income_df["value"],
            errors="coerce",
        )

        income_df = income_df.dropna(
            subset=["date", "value"]
        )

        report_dates = sorted(
            pd.Timestamp(item)
            for item in income_df["date"].dropna().unique()
        )

        # 最近 3 個已公告季度
        latest_3_quarters = report_dates[-3:]

        if len(latest_3_quarters) != 3:
            return result

        revenue_values = [
            self._pick_type_value(
                income_df,
                report_date,
                ["Revenue"],
            )
            for report_date in latest_3_quarters
        ]

        net_income_values = [
            self._pick_type_value(
                income_df,
                report_date,
                [
                    "NetIncome",
                    "IncomeAfterTaxes",
                ],
            )
            for report_date in latest_3_quarters
        ]

        if (
            any(value is None for value in revenue_values)
            or any(value is None for value in net_income_values)
        ):
            return result

        revenue_3q = float(sum(revenue_values))
        net_income_3q = float(sum(net_income_values))

        weighted_net_margin_3q = safe_divide(
            net_income_3q,
            revenue_3q,
        )

        if weighted_net_margin_3q is None:
            return result

        # 防止資料異常，正常非金融公司可容許到 60%
        weighted_net_margin_3q = max(
            0.0,
            min(0.60, weighted_net_margin_3q),
        )

        result["weighted_net_margin_3q"] = (
            weighted_net_margin_3q
        )

        # ---------------------------------------------------------
        # 3. 最近四季平均股數
        # ---------------------------------------------------------
        fundamentals_df = self.get_historical_fundamentals(
            ticker,
            start_date,
        )

        if (
            fundamentals_df is None
            or fundamentals_df.empty
            or "shares_outstanding" not in fundamentals_df.columns
        ):
            return result

        valid_shares = (
            fundamentals_df
            .dropna(subset=["shares_outstanding"])
            .sort_values("report_date")
        )

        valid_shares = valid_shares[
            pd.to_numeric(
                valid_shares["shares_outstanding"],
                errors="coerce",
            ) > 0
        ]

        recent_shares = valid_shares.tail(4)[
            "shares_outstanding"
        ]

        if len(recent_shares) < 2:
            return result

        average_shares_12m = float(
            pd.to_numeric(
                recent_shares,
                errors="coerce",
            ).mean()
        )

        if average_shares_12m <= 0:
            return result

        result["average_shares_12m"] = average_shares_12m

        # ---------------------------------------------------------
        # 4. 最終 EPS TTM Nowcast
        # ---------------------------------------------------------
        estimated_net_income = (
            ltm_month_revenue
            * weighted_net_margin_3q
        )

        eps_ttm_nowcast = safe_divide(
            estimated_net_income,
            average_shares_12m,
        )

        result["eps_ttm_nowcast"] = eps_ttm_nowcast

        return result
    
    
    
    def get_cashflow_history(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, list[float]]:
        """
        建立台股現金流歷史資料。

        FinMind 現金流量表中的：
        - CashFlowsFromOperatingActivities：年初至今累積營業現金流
        - PropertyAndPlantAndEquipment：年初至今累積資本支出，通常為負數

        本函式會轉成單季資料，並回傳最新到最舊的序列。

        回傳：
        {
            "quarterly_operating_cash_flow": [...],
            "quarterly_capex": [...],            # 正數，代表支出
            "quarterly_free_cash_flow": [...],
            "operating_cash_flow": [...],        # 完整年度 OCF
            "capex": [...],                      # 完整年度 CapEx，正數
            "free_cash_flow": [...],             # 完整年度 FCF
        }
        """

        empty_history = {
            "quarterly_operating_cash_flow": [],
            "quarterly_capex": [],
            "quarterly_free_cash_flow": [],
            "operating_cash_flow": [],
            "capex": [],
            "free_cash_flow": [],
        }

        if not self.is_taiwan_ticker(ticker):
            return empty_history

        cashflow_df = self.get_cash_flow_statement(
            ticker,
            start_date,
        )

        required_cols = {"date", "type", "value"}

        if (
            cashflow_df is None
            or cashflow_df.empty
            or not required_cols.issubset(cashflow_df.columns)
        ):
            LOGGER.warning(
                "FinMind cashflow history unavailable for %s",
                ticker,
            )
            return empty_history

        cashflow_df = cashflow_df.copy()

        cashflow_df["date"] = pd.to_datetime(
            cashflow_df["date"],
            errors="coerce",
        )

        cashflow_df["value"] = pd.to_numeric(
            cashflow_df["value"],
            errors="coerce",
        )

        cashflow_df = cashflow_df.dropna(
            subset=["date", "value"]
        )

        if cashflow_df.empty:
            return empty_history

        # FinMind 已確認的主要欄位。
        # 若未來某公司有不同名稱，可於此補候選名稱。
        ocf_candidates = [
            "CashFlowsFromOperatingActivities",
            "NetCashInflowFromOperatingActivities",
        ]

        capex_candidates = [
            "PropertyAndPlantAndEquipment",
        ]

        report_dates = sorted(
            pd.Timestamp(item)
            for item in cashflow_df["date"].dropna().unique()
        )

        cumulative_ocf = [
            self._pick_type_value(
                cashflow_df,
                report_date,
                ocf_candidates,
            )
            for report_date in report_dates
        ]

        cumulative_capex_raw = [
            self._pick_type_value(
                cashflow_df,
                report_date,
                capex_candidates,
            )
            for report_date in report_dates
        ]

        # 現金流為累積值，轉為單季值。
        quarterly_ocf = self._quarterly_from_cumulative(
            cumulative_ocf,
            report_dates,
        )

        quarterly_capex_signed = self._quarterly_from_cumulative(
            cumulative_capex_raw,
            report_dates,
        )

        # PropertyAndPlantAndEquipment 通常為負數，轉成正的 CapEx 支出。
        quarterly_capex = [
            abs(value) if value is not None else None
            for value in quarterly_capex_signed
        ]

        quarterly_fcf = []

        for ocf, capex in zip(
            quarterly_ocf,
            quarterly_capex,
        ):
            if ocf is None or capex is None:
                quarterly_fcf.append(None)
            else:
                quarterly_fcf.append(float(ocf - capex))

        # ---------------------------------------------------------
        # 建立完整年度 OCF / CapEx / FCF
        # 只採用有完整 Q1~Q4 的年度，避免當年度尚未結束而誤算
        # ---------------------------------------------------------
        quarter_df = pd.DataFrame(
            {
                "date": report_dates,
                "quarterly_ocf": quarterly_ocf,
                "quarterly_capex": quarterly_capex,
                "quarterly_fcf": quarterly_fcf,
            }
        )

        quarter_df["year"] = quarter_df["date"].dt.year
        quarter_df["quarter"] = quarter_df["date"].dt.quarter

        annual_rows: list[dict[str, Any]] = []

        for year, group in quarter_df.groupby("year"):
            group = group.sort_values("quarter")

            available_quarters = set(
                group["quarter"]
                .dropna()
                .astype(int)
                .tolist()
            )

            # 只有完整年度才納入 CAGR 歷史
            if available_quarters != {1, 2, 3, 4}:
                continue

            ocf_values = group["quarterly_ocf"].dropna()
            capex_values = group["quarterly_capex"].dropna()
            fcf_values = group["quarterly_fcf"].dropna()

            annual_rows.append(
                {
                    "year": int(year),
                    "operating_cash_flow": (
                        float(ocf_values.sum())
                        if len(ocf_values) == 4
                        else None
                    ),
                    "capex": (
                        float(capex_values.sum())
                        if len(capex_values) == 4
                        else None
                    ),
                    "free_cash_flow": (
                        float(fcf_values.sum())
                        if len(fcf_values) == 4
                        else None
                    ),
                }
            )

        annual_df = pd.DataFrame(annual_rows)

        if annual_df.empty:
            annual_ocf = []
            annual_capex = []
            annual_fcf = []
        else:
            annual_df = annual_df.sort_values(
                "year",
                ascending=False,
            )

            annual_ocf = (
                annual_df["operating_cash_flow"]
                .dropna()
                .astype(float)
                .tolist()
            )

            annual_capex = (
                annual_df["capex"]
                .dropna()
                .astype(float)
                .tolist()
            )

            annual_fcf = (
                annual_df["free_cash_flow"]
                .dropna()
                .astype(float)
                .tolist()
            )

        # QualityEngine 的歷史資料假設：最新 → 最舊
        return {
            "quarterly_operating_cash_flow": [
                float(value)
                for value in reversed(quarterly_ocf)
                if value is not None
            ],
            "quarterly_capex": [
                float(value)
                for value in reversed(quarterly_capex)
                if value is not None
            ],
            "quarterly_free_cash_flow": [
                float(value)
                for value in reversed(quarterly_fcf)
                if value is not None
            ],
            "operating_cash_flow": annual_ocf,
            "capex": annual_capex,
            "free_cash_flow": annual_fcf,
        }

    def get_snapshot_financial_fields(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, float | None]:
        """
        建立可覆蓋 FinancialSnapshot 的最新財務欄位。

        損益表類欄位採最近四季合計 TTM：
        - revenue
        - gross_profit
        - operating_income
        - net_income
        - interest_expense（若可取得）

        資產負債表類欄位採最新可得季度：
        - total_equity
        - cash
        - total_debt
        - current_assets
        - current_liabilities
        - inventory

        注意：
        若某欄位資料不足，回傳 None，不會覆蓋 Yahoo fallback。
        """

        result = {
            "revenue": None,
            "gross_profit": None,
            "operating_income": None,
            "net_income": None,
            "interest_expense": None,
            "total_equity": None,
            "cash": None,
            "total_debt": None,
            "current_assets": None,
            "current_liabilities": None,
            "inventory": None,
        }

        if not self.is_taiwan_ticker(ticker):
            return result

        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        balance_df = self.get_balance_sheet(
            ticker,
            start_date,
        )

        required_cols = {"date", "type", "value"}

        # ---------------------------------------------------------
        # 損益表：最近四季 TTM
        # ---------------------------------------------------------
        if (
            income_df is not None
            and not income_df.empty
            and required_cols.issubset(income_df.columns)
        ):
            income_df = income_df.copy()

            income_df["date"] = pd.to_datetime(
                income_df["date"],
                errors="coerce",
            )

            income_df["value"] = pd.to_numeric(
                income_df["value"],
                errors="coerce",
            )

            income_df = income_df.dropna(
                subset=["date", "value"]
            )

            if not income_df.empty:
                # 使用已驗證為單季值的損益表項目
                metric_candidates = {
                    "revenue": [
                        "Revenue",
                    ],
                    "gross_profit": [
                        "GrossProfit",
                    ],
                    "operating_income": [
                        "OperatingIncome",
                    ],
                    "net_income": [
                        "NetIncome",
                        "IncomeAfterTaxes",
                    ],
                    # FinMind 不同公司可能未提供此欄；缺值即保留 Yahoo
                    "interest_expense": [
                        "InterestExpense",
                        "FinanceCosts",
                        "InterestAndFinanceCosts",
                    ],
                }

                report_dates = sorted(
                    pd.Timestamp(item)
                    for item in income_df["date"].dropna().unique()
                )

                # 最近四個財報季度
                latest_four_dates = report_dates[-4:]

                if len(latest_four_dates) == 4:
                    for field_name, candidates in metric_candidates.items():
                        values = [
                            self._pick_type_value(
                                income_df,
                                report_date,
                                candidates,
                            )
                            for report_date in latest_four_dates
                        ]

                        # TTM 必須四季都有資料，避免用部分年度資料誤算
                        if all(value is not None for value in values):
                            result[field_name] = float(sum(values))

        # ---------------------------------------------------------
        # 資產負債表：最新可得季度
        # ---------------------------------------------------------
        if (
            balance_df is not None
            and not balance_df.empty
            and required_cols.issubset(balance_df.columns)
        ):
            balance_df = balance_df.copy()

            balance_df["date"] = pd.to_datetime(
                balance_df["date"],
                errors="coerce",
            )

            balance_df["value"] = pd.to_numeric(
                balance_df["value"],
                errors="coerce",
            )

            balance_df = balance_df.dropna(
                subset=["date", "value"]
            )

            if not balance_df.empty:
                latest_balance_date = balance_df["date"].max()

                balance_candidates = {
                    "total_equity": [
                        "EquityAttributableToOwnersOfParent",
                        "Equity",
                    ],
                    "cash": [
                        "CashAndCashEquivalents",
                    ],
                    "current_assets": [
                        "CurrentAssets",
                    ],
                    "current_liabilities": [
                        "CurrentLiabilities",
                    ],
                    "inventory": [
                        "Inventories",
                        "Inventory",
                    ],
                }

                for field_name, candidates in balance_candidates.items():
                    result[field_name] = self._pick_type_value(
                        balance_df,
                        latest_balance_date,
                        candidates,
                    )

                # 台股財報中的 Total Debt 不一定有統一欄位，
                # 先以短債 + 長債 + 公司債作簡化估計。
                short_debt = self._pick_type_value(
                    balance_df,
                    latest_balance_date,
                    [
                        "ShorttermBorrowings",
                        "ShortTermBorrowings",
                    ],
                )

                long_debt = self._pick_type_value(
                    balance_df,
                    latest_balance_date,
                    [
                        "LongtermBorrowings",
                        "LongTermBorrowings",
                    ],
                )

                bonds_payable = self._pick_type_value(
                    balance_df,
                    latest_balance_date,
                    [
                        "BondsPayable",
                    ],
                )

                debt_values = [
                    value
                    for value in [
                        short_debt,
                        long_debt,
                        bonds_payable,
                    ]
                    if value is not None
                ]

                if debt_values:
                    result["total_debt"] = float(sum(debt_values))

        return result
    
    @staticmethod
    def _pick_type_value(
        df: pd.DataFrame,
        report_date: pd.Timestamp,
        candidates: list[str],
    ) -> float | None:
        """
        從 FinMind 長格式資料中，
        尋找指定日期與候選 type 名稱的第一個有效數值。
        """

        if df is None or df.empty:
            return None

        required_cols = {"date", "type", "value"}
        if not required_cols.issubset(df.columns):
            return None

        date_value = pd.Timestamp(report_date).normalize()

        subset = df[
            df["date"].dt.normalize() == date_value
        ]

        for item_type in candidates:
            match = subset[
                subset["type"] == item_type
            ]

            if match.empty:
                continue

            values = pd.to_numeric(
                match["value"],
                errors="coerce",
            ).dropna()

            if not values.empty:
                return float(values.iloc[0])

        return None

    @staticmethod
    def _quarterly_from_cumulative(
        cumulative_values: list[float | None],
        dates: list[pd.Timestamp],
    ) -> list[float | None]:
        """
        把台灣常見的「年初至今累積 EPS」轉成單季 EPS。

        輸入必須按時間由舊到新排序。

        規則：
        - 新年度第一筆：直接視為該年度第一個季度的 EPS
        - 同年度後續資料：本期累積 EPS - 前期累積 EPS
        """

        quarterly_values: list[float | None] = []

        previous_cumulative: float | None = None
        previous_year: int | None = None

        for report_date, cumulative in zip(dates, cumulative_values):
            current_year = pd.Timestamp(report_date).year

            if cumulative is None:
                quarterly_values.append(None)
                previous_cumulative = None
                previous_year = current_year
                continue

            # 新年度第一筆，或上一筆資料無效
            if previous_year != current_year or previous_cumulative is None:
                quarterly_values.append(float(cumulative))
            else:
                quarterly_values.append(
                    float(cumulative - previous_cumulative)
                )

            previous_cumulative = float(cumulative)
            previous_year = current_year

        return quarterly_values

    def _shares_from_capital(
        self,
        ticker: str,
        balance_df: pd.DataFrame,
        report_date: pd.Timestamp,
    ) -> float | None:
        """
        以股本 / 面額推估期末發行股數。

        FinMind 的 CapitalStock / OrdinaryShare 通常為股本金額，
        非直接股數，因此以：
            shares = capital_stock / par_value
        推估。

        台灣大多數公司面額為 10 元。
        """

        capital_candidates = [
            "CapitalStock",
            "OrdinaryShare",
        ]

        capital_stock = self._pick_type_value(
            balance_df,
            report_date,
            capital_candidates,
        )

        par_value = self.ticker_par_values.get(
            ticker,
            self.default_par_value,
        )

        if capital_stock is None or par_value in (None, 0):
            return None

        return safe_divide(capital_stock, par_value)

    def get_historical_fundamentals(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """
        回傳標準化歷史基本面資料。

        回傳欄位：
        - report_date
        - ttm_eps
        - bvps

        額外保留欄位供 debug：
        - quarterly_eps
        - cumulative_eps
        - total_equity
        - shares_outstanding
        """
        
        columns = [
            "report_date",
            "ttm_eps",
            "bvps",
            "quarterly_eps",
            "reported_eps",
            "total_equity",
            "shares_outstanding",
        ]

        empty = pd.DataFrame(columns=columns)

        if not self.is_taiwan_ticker(ticker):
            return empty

        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        balance_df = self.get_balance_sheet(
            ticker,
            start_date,
        )

        if income_df.empty:
            LOGGER.warning(
                "FinMind income statement is empty for %s",
                ticker,
            )
            return empty

        required_cols = {"date", "type", "value"}
        if not required_cols.issubset(income_df.columns):
            LOGGER.warning(
                "FinMind income data format invalid for %s",
                ticker,
            )
            return empty

        income_df = income_df.copy()
        income_df["date"] = pd.to_datetime(
            income_df["date"],
            errors="coerce",
        )
        income_df["value"] = pd.to_numeric(
            income_df["value"],
            errors="coerce",
        )
        income_df = income_df.dropna(
            subset=["date"]
        )

        if not balance_df.empty:
            if required_cols.issubset(balance_df.columns):
                balance_df = balance_df.copy()
                balance_df["date"] = pd.to_datetime(
                    balance_df["date"],
                    errors="coerce",
                )
                balance_df["value"] = pd.to_numeric(
                    balance_df["value"],
                    errors="coerce",
                )
                balance_df = balance_df.dropna(
                    subset=["date"]
                )
            else:
                LOGGER.warning(
                    "FinMind balance sheet format invalid for %s",
                    ticker,
                )
                balance_df = pd.DataFrame()

        report_dates = sorted(
            pd.Timestamp(item)
            for item in income_df["date"].dropna().unique()
        )

        # 依剛剛測試結果，FinMind 已確認 type 為 EPS。
        eps_candidates = [
            "EPS",
        ]

        # 優先使用歸屬母公司業主權益，較適合每股淨值。
        equity_candidates = [
            "EquityAttributableToOwnersOfParent",
            "Equity",
        ]
        
        # FinMind TaiwanStockFinancialStatements 的 EPS 經驗證後為「單季 EPS」。
        # 不要再當作累積 EPS 做差分。
        reported_eps = [
            self._pick_type_value(
                income_df,
                report_date,
                eps_candidates,
            )
            for report_date in report_dates
        ]

        quarterly_eps = reported_eps.copy()
        
        rows: list[dict[str, Any]] = []

        for index, report_date in enumerate(report_dates):
            # TTM EPS = 最近四季單季 EPS 加總
            ttm_eps = None

            if index >= 3:
                eps_window = quarterly_eps[
                    index - 3:index + 1
                ]

                if all(value is not None for value in eps_window):
                    ttm_eps = float(sum(eps_window))

            total_equity = None
            shares_outstanding = None
            bvps = None

            if not balance_df.empty:
                total_equity = self._pick_type_value(
                    balance_df,
                    report_date,
                    equity_candidates,
                )

                shares_outstanding = self._shares_from_capital(
                    ticker,
                    balance_df,
                    report_date,
                )

                bvps = safe_divide(
                    total_equity,
                    shares_outstanding,
                )
                        
            rows.append(
                {
                    "report_date": report_date,
                    "ttm_eps": ttm_eps,
                    "bvps": bvps,
                    "quarterly_eps": quarterly_eps[index],
                    "reported_eps": reported_eps[index],
                    "total_equity": total_equity,
                    "shares_outstanding": shares_outstanding,
                }
            )

        result = pd.DataFrame(rows)

        if result.empty:
            return empty

        result = (
            result
            .sort_values("report_date")
            .reset_index(drop=True)
        )

        valid_ttm_eps = result["ttm_eps"].notna().sum()
        valid_bvps = result["bvps"].notna().sum()

        LOGGER.info(
            "FinMind fundamentals built for %s: TTM EPS=%s, BVPS=%s",
            ticker,
            valid_ttm_eps,
            valid_bvps,
        )

        return result
    

    def get_growth_history(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, list[float]]:
        """
        建立給 QualityEngine 使用的成長資料。

        回傳格式與既有 snapshot.financial_history 相容：

        {
            "quarterly_revenue": [...],  # 最新到最舊，單季營收
            "quarterly_eps": [...],      # 最新到最舊，單季 EPS
            "revenue": [...],            # 最新到最舊，完整年度營收
            "eps": [...],                # 最新到最舊，完整年度 EPS
        }

        注意：
        - FinMind 的 EPS 已驗證為單季 EPS
        - Revenue 需先用 Step 0 驗證為單季營收
        - 年度資料只採用完整四季，避免 2026 Q1 被誤當成年營收
        """

        empty_history = {
            "quarterly_revenue": [],
            "quarterly_eps": [],
            "revenue": [],
            "eps": [],
        }

        if not self.is_taiwan_ticker(ticker):
            return empty_history

        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        required_cols = {"date", "type", "value"}

        if (
            income_df is None
            or income_df.empty
            or not required_cols.issubset(income_df.columns)
        ):
            LOGGER.warning(
                "FinMind growth history unavailable for %s",
                ticker,
            )
            return empty_history

        income_df = income_df.copy()

        income_df["date"] = pd.to_datetime(
            income_df["date"],
            errors="coerce",
        )

        income_df["value"] = pd.to_numeric(
            income_df["value"],
            errors="coerce",
        )

        income_df = income_df.dropna(
            subset=["date", "value"]
        )

        if income_df.empty:
            return empty_history

        # 將長格式轉成每季一列的寬格式
        quarterly = (
            income_df[
                income_df["type"].isin(["Revenue", "EPS"])
            ]
            .pivot_table(
                index="date",
                columns="type",
                values="value",
                aggfunc="first",
            )
            .reset_index()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if quarterly.empty:
            return empty_history

        # 確保欄位存在，避免某些公司缺 Revenue 或 EPS 時報錯
        if "Revenue" not in quarterly.columns:
            quarterly["Revenue"] = pd.NA

        if "EPS" not in quarterly.columns:
            quarterly["EPS"] = pd.NA

        quarterly["Revenue"] = pd.to_numeric(
            quarterly["Revenue"],
            errors="coerce",
        )

        quarterly["EPS"] = pd.to_numeric(
            quarterly["EPS"],
            errors="coerce",
        )

        # ------------------------------------------------------
        # 季資料：最新 → 最舊
        # QualityEngine._period_growth() 假設 values[0] 為最新一期
        # ------------------------------------------------------
        quarterly_desc = quarterly.sort_values(
            "date",
            ascending=False,
        ).reset_index(drop=True)

        quarterly_revenue = (
            quarterly_desc["Revenue"]
            .dropna()
            .astype(float)
            .tolist()
        )

        quarterly_eps = (
            quarterly_desc["EPS"]
            .dropna()
            .astype(float)
            .tolist()
        )

        # ------------------------------------------------------
        # 年資料：只有完整 4 季才納入
        # 防止目前年度例如 2026 年只有 Q1，被誤判為完整年度資料
        # ------------------------------------------------------
        quarterly["year"] = quarterly["date"].dt.year
        quarterly["quarter"] = quarterly["date"].dt.quarter

        annual_rows: list[dict[str, Any]] = []

        for year, group in quarterly.groupby("year"):
            group = group.sort_values("quarter")

            available_quarters = set(
                group["quarter"].dropna().astype(int).tolist()
            )

            # 只採完整 Q1/Q2/Q3/Q4 年度
            if available_quarters != {1, 2, 3, 4}:
                continue

            revenue_values = group["Revenue"].dropna()
            eps_values = group["EPS"].dropna()

            annual_revenue = (
                float(revenue_values.sum())
                if len(revenue_values) == 4
                else None
            )

            annual_eps = (
                float(eps_values.sum())
                if len(eps_values) == 4
                else None
            )

            annual_rows.append(
                {
                    "year": int(year),
                    "revenue": annual_revenue,
                    "eps": annual_eps,
                }
            )

        annual_df = pd.DataFrame(annual_rows)

        if annual_df.empty:
            annual_revenue = []
            annual_eps = []
        else:
            annual_df = annual_df.sort_values(
                "year",
                ascending=False,
            )

            annual_revenue = (
                annual_df["revenue"]
                .dropna()
                .astype(float)
                .tolist()
            )

            annual_eps = (
                annual_df["eps"]
                .dropna()
                .astype(float)
                .tolist()
            )

        return {
            "quarterly_revenue": quarterly_revenue,
            "quarterly_eps": quarterly_eps,
            "revenue": annual_revenue,
            "eps": annual_eps,
        }
    

class TaiwanChipDataProvider(YahooFinanceProvider):
    def get_chip_data(self, ticker: str) -> pd.DataFrame:
        if not ticker.endswith((".TW", ".TWO")):
            return super().get_chip_data(ticker)

        try:
            import requests
        except ImportError as exc:
            raise ImportError("Install requests to use Taiwan chip data: pip install requests") from exc

        code = ticker.split(".")[0]
        if ticker.endswith(".TW"):
            return self._twse_chip_history(code, requests)
        return self._tpex_chip_history(code, requests)

    def _twse_chip_history(self, code: str, requests: object) -> pd.DataFrame:
        rows: list[dict[str, object]] = []

        for day in pd.bdate_range(end=pd.Timestamp.today(), periods=10):
            try:
                response = requests.get(
                    "https://www.twse.com.tw/rwd/zh/fund/T86",
                    params={"date": day.strftime("%Y%m%d"), "selectType": "ALLBUT0999", "response": "json"},
                    headers={"User-Agent": "Mozilla/5.0"},
                    timeout=15,
                )
            except Exception as exc:
                LOGGER.warning("TWSE chip request failed for %s on %s: %s", code, day.date(), exc)
                continue

            if not response.ok:
                continue

            try:
                payload = response.json()
            except Exception as exc:
                LOGGER.warning("TWSE chip JSON parse failed for %s on %s: %s", code, day.date(), exc)
                continue

            data = payload.get("data", [])
            match = next((record for record in data if str(record[0]).strip() == code), None)

            if match:
                rows.append({
                    "date": day,
                    "institutional_net_buy": self._number(match[-1]),
                    "margin_balance": None,
                })

        return pd.DataFrame(rows).sort_values("date") if rows else pd.DataFrame(columns=["date", "institutional_net_buy", "margin_balance"])

    def _tpex_chip_history(self, code: str, requests: object) -> pd.DataFrame:
        response = requests.get(
            "https://www.tpex.org.tw/openapi/v1/tpex_3insti_daily_trading",
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15,
        )
        if not response.ok:
            return pd.DataFrame(columns=["date", "institutional_net_buy", "margin_balance"])

        records = response.json()
        match = next(
            (record for record in records if str(record.get("SecuritiesCompanyCode", next(iter(record.values()), ""))).strip() == code),
            None,
        )
        if not match:
            return pd.DataFrame(columns=["date", "institutional_net_buy", "margin_balance"])

        net = next(
            (value for key, value in match.items() if "net" in key.lower() and ("total" in key.lower() or "buy" in key.lower())),
            None,
        )

        return pd.DataFrame([{
            "date": pd.Timestamp.today(),
            "institutional_net_buy": self._number(net),
            "margin_balance": None,
        }])

    @staticmethod
    def _number(value: object) -> float | None:
        if value is None:
            return None
        try:
            return float(str(value).replace(",", "").replace("+", ""))
        except ValueError:
            return None
        
        
class HybridTaiwanProvider(TaiwanChipDataProvider):
    """
    台股：
      Yahoo Finance → 現價、歷史價格、成交量、市值
      FinMind       → 歷史 TTM EPS、BVPS、當前財報基本面
      TWSE / TPEX   → 法人籌碼

    美股：
      Yahoo Finance → 價格與基本面
    """

    def __init__(self, config: dict[str, Any]) -> None:
        super().__init__()
        self.finmind = FinMindFundamentalProvider(config)

        # 避免同一檔股票在一次分析中重複抓 FinMind API
        self._finmind_fundamentals_cache: dict[str, pd.DataFrame] = {}

    @staticmethod
    def _is_taiwan_ticker(ticker: str) -> bool:
        return ticker.endswith((".TW", ".TWO"))

    def _get_finmind_fundamentals(
        self,
        ticker: str,
    ) -> pd.DataFrame:
        """
        取得 FinMind 歷史基本面，並在同一次 notebook/session 中快取。
        """
        if ticker not in self._finmind_fundamentals_cache:
            self._finmind_fundamentals_cache[ticker] = (
                self.finmind.get_historical_fundamentals(ticker)
            )

        return self._finmind_fundamentals_cache[ticker].copy()

    def get_snapshot(self, ticker: str) -> FinancialSnapshot:
        """
        資料來源分工：

        台股：
        - Yahoo    ：現價、價格歷史、成交量、市值等市場資料
        - FinMind  ：TTM EPS、BVPS、股數、營收/EPS/現金流歷史
        - TWSE/TPEX：法人籌碼資料

        美股：
        - Yahoo：維持原本完整資料流程
        """

        # Yahoo 作為所有欄位的初始值與 fallback
        snapshot = super().get_snapshot(ticker)

        # 美股不使用 FinMind
        if not self._is_taiwan_ticker(ticker):
            return snapshot

        finmind_enriched = False

        # ---------------------------------------------------------
        # 第一階段：FinMind TTM EPS / BVPS / 股數
        # ---------------------------------------------------------
        try:
            finmind_df = self._get_finmind_fundamentals(ticker)

            if finmind_df is not None and not finmind_df.empty:
                finmind_df = (
                    finmind_df
                    .sort_values("report_date")
                    .reset_index(drop=True)
                )

                # 最新有效 TTM EPS
                if "ttm_eps" in finmind_df.columns:
                    valid_eps = finmind_df.dropna(subset=["ttm_eps"])

                    if not valid_eps.empty:
                        latest_ttm_eps = float(
                            valid_eps.iloc[-1]["ttm_eps"]
                        )

                        if latest_ttm_eps > 0:
                            snapshot.ttm_eps = latest_ttm_eps
                            snapshot.eps = latest_ttm_eps
                            finmind_enriched = True

                # 最新有效 BVPS
                if "bvps" in finmind_df.columns:
                    valid_bvps = finmind_df.dropna(subset=["bvps"])

                    if not valid_bvps.empty:
                        latest_bvps = float(
                            valid_bvps.iloc[-1]["bvps"]
                        )

                        if latest_bvps > 0:
                            snapshot.book_value_per_share = latest_bvps
                            snapshot.book_value = latest_bvps
                            finmind_enriched = True
                            
                # 最新股數與近四季平均股數
                if "shares_outstanding" in finmind_df.columns:
                    valid_shares = (
                        finmind_df
                        .dropna(subset=["shares_outstanding"])
                        .sort_values("report_date")
                        .copy()
                    )

                    valid_shares["shares_outstanding"] = pd.to_numeric(
                        valid_shares["shares_outstanding"],
                        errors="coerce",
                    )

                    valid_shares = valid_shares[
                        valid_shares["shares_outstanding"] > 0
                    ]

                    if not valid_shares.empty:
                        # 最新股數：一般財務欄位仍使用最新值
                        snapshot.shares_outstanding = float(
                            valid_shares.iloc[-1]["shares_outstanding"]
                        )

                        # EPS TTM Nowcast 專用：近四季平均股數
                        recent_shares = valid_shares.tail(4)[
                            "shares_outstanding"
                        ]

                        snapshot.average_shares_12m = float(
                            recent_shares.mean()
                        )

                        finmind_enriched = True
                            
        except Exception as exc:
            LOGGER.warning(
                "FinMind historical fundamentals enrichment failed for %s: %s. "
                "Keeping Yahoo fields as fallback.",
                ticker,
                exc,
            )

        # ---------------------------------------------------------
        # 第二階段 A：FinMind 營收 / EPS 歷史與季度成長資料
        # ---------------------------------------------------------
        try:
            growth_history = self.finmind.get_growth_history(ticker)

            # 只在 FinMind 有資料時覆蓋 Yahoo，
            # 避免空 list 蓋掉 Yahoo fallback。
            for history_key in [
                "quarterly_revenue",
                "quarterly_eps",
                "revenue",
                "eps",
            ]:
                values = growth_history.get(history_key, [])

                if values:
                    snapshot.financial_history[history_key] = values
                    finmind_enriched = True

        except Exception as exc:
            LOGGER.warning(
                "FinMind growth history enrichment failed for %s: %s. "
                "Keeping Yahoo financial history as fallback.",
                ticker,
                exc,
            )

        # ---------------------------------------------------------
        # 第二階段 B：FinMind 現金流資料
        # ---------------------------------------------------------
        try:
            cashflow_history = self.finmind.get_cashflow_history(ticker)

            # 年度歷史資料：供 QualityEngine CAGR 使用
            for history_key in [
                "operating_cash_flow",
                "capex",
                "free_cash_flow",
            ]:
                values = cashflow_history.get(history_key, [])

                if values:
                    snapshot.financial_history[history_key] = values
                    finmind_enriched = True

            # 季度歷史資料：供日後分析、除錯或擴充使用
            for history_key in [
                "quarterly_operating_cash_flow",
                "quarterly_capex",
                "quarterly_free_cash_flow",
            ]:
                values = cashflow_history.get(history_key, [])

                if values:
                    snapshot.financial_history[history_key] = values
                    finmind_enriched = True

            quarterly_ocf = cashflow_history.get(
                "quarterly_operating_cash_flow",
                [],
            )

            quarterly_capex = cashflow_history.get(
                "quarterly_capex",
                [],
            )

            quarterly_fcf = cashflow_history.get(
                "quarterly_free_cash_flow",
                [],
            )

            # 最近四季 TTM OCF
            if len(quarterly_ocf) >= 4:
                snapshot.operating_cash_flow = float(
                    sum(quarterly_ocf[:4])
                )
                finmind_enriched = True

            # 最近四季 TTM CapEx
            # FinMind Provider 中已轉為正數，代表資本支出金額。
            if len(quarterly_capex) >= 4:
                snapshot.capex = float(
                    sum(quarterly_capex[:4])
                )
                finmind_enriched = True

            # 最近四季 TTM FCF
            if len(quarterly_fcf) >= 4:
                snapshot.free_cash_flow = float(
                    sum(quarterly_fcf[:4])
                )
                finmind_enriched = True

        except Exception as exc:
            LOGGER.warning(
                "FinMind cashflow history enrichment failed for %s: %s. "
                "Keeping Yahoo cashflow fields as fallback.",
                ticker,
                exc,
            )

            
        # ---------------------------------------------------------
        # 第二階段 C：
        # FinMind 最新 TTM 損益表與最新資產負債表欄位
        # ---------------------------------------------------------
        try:
            financial_fields = self.finmind.get_snapshot_financial_fields(
                ticker
            )

            # 僅以有效 FinMind 數值覆蓋 Yahoo；
            # FinMind 缺欄位時仍保留 Yahoo fallback。
            for field_name, value in financial_fields.items():
                if (
                    value is not None
                    and hasattr(snapshot, field_name)
                ):
                    setattr(snapshot, field_name, float(value))
                    finmind_enriched = True

        except Exception as exc:
            LOGGER.warning(
                "FinMind snapshot financial fields enrichment failed for %s: %s. "
                "Keeping Yahoo snapshot fields as fallback.",
                ticker,
                exc,
            )

        # ---------------------------------------------------------
        # 第二階段 D：
        # FinMind 月營收推估「即時 TTM EPS Nowcast」
        #
        # 公式：
        # 最近12個已公告月營收
        # × 近三季營收加權平均淨利率
        # ÷ 近四季平均股數
        # ---------------------------------------------------------
        try:
            nowcast = self.finmind.get_eps_ttm_nowcast_inputs(
                ticker
            )

            eps_ttm_nowcast = nowcast.get(
                "eps_ttm_nowcast"
            )

            if (
                eps_ttm_nowcast is not None
                and eps_ttm_nowcast > 0
            ):
                snapshot.eps_ttm_nowcast = float(
                    eps_ttm_nowcast
                )

                snapshot.pe_ttm_nowcast = safe_divide(
                    snapshot.current_price,
                    snapshot.eps_ttm_nowcast,
                )

                finmind_enriched = True

        except Exception as exc:
            LOGGER.warning(
                "FinMind EPS TTM Nowcast failed for %s: %s. "
                "Keeping Nowcast empty.",
                ticker,
                exc,
            )
            
        # ---------------------------------------------------------
        # 資料來源標示
        # ---------------------------------------------------------
        if finmind_enriched:
            snapshot.source = (
                "Yahoo Price + FinMind Fundamentals + TWSE/TPEX Flow"
            )

        return snapshot
    
    def get_historical_fundamentals(
        self,
        ticker: str,
    ) -> pd.DataFrame:
        """
        台股優先使用 FinMind 的 TTM EPS / BVPS。
        資料不足時 fallback 回 Yahoo。

        美股完全維持 Yahoo。
        """
        if not self._is_taiwan_ticker(ticker):
            return super().get_historical_fundamentals(ticker)

        try:
            finmind_df = self._get_finmind_fundamentals(ticker)

            if finmind_df is not None and not finmind_df.empty:
                eps_count = (
                    finmind_df["ttm_eps"].notna().sum()
                    if "ttm_eps" in finmind_df.columns
                    else 0
                )

                bvps_count = (
                    finmind_df["bvps"].notna().sum()
                    if "bvps" in finmind_df.columns
                    else 0
                )

                # 至少 4 個有效季度的 TTM EPS/BVPS 才採用
                if eps_count >= 4 and bvps_count >= 4:
                    LOGGER.info(
                        "Using FinMind historical fundamentals for %s "
                        "(TTM EPS=%s, BVPS=%s)",
                        ticker,
                        eps_count,
                        bvps_count,
                    )

                    return (
                        finmind_df[
                            ["report_date", "ttm_eps", "bvps"]
                        ]
                        .dropna(
                            how="all",
                            subset=["ttm_eps", "bvps"],
                        )
                        .sort_values("report_date")
                        .reset_index(drop=True)
                    )

        except Exception as exc:
            LOGGER.warning(
                "FinMind historical fundamentals failed for %s: %s; "
                "falling back to Yahoo.",
                ticker,
                exc,
            )

        return super().get_historical_fundamentals(ticker)
    
# =========================
# Quality
# =========================
@dataclass
class QualityMetrics:
    roe: float | None = None
    roic: float | None = None
    gross_margin: float | None = None
    operating_margin: float | None = None
    net_margin: float | None = None
    debt_to_equity: float | None = None
    interest_coverage: float | None = None
    current_ratio: float | None = None
    quick_ratio: float | None = None
    fcf_margin: float | None = None
    revenue_cagr_1y: float | None = None    
    revenue_cagr_3y: float | None = None
    revenue_cagr_5y: float | None = None
    revenue_cagr_10y: float | None = None
    eps_cagr_1y: float | None = None
    eps_cagr_3y: float | None = None
    eps_cagr_5y: float | None = None
    eps_cagr_10y: float | None = None
    fcf_cagr_1y: float | None = None
    fcf_cagr_3y: float | None = None
    fcf_cagr_5y: float | None = None
    fcf_cagr_10y: float | None = None
    revenue_growth_3m: float | None = None
    revenue_growth_6m: float | None = None
    revenue_growth_9m: float | None = None
    eps_growth_3m: float | None = None
    eps_growth_6m: float | None = None
    eps_growth_9m: float | None = None


class QualityEngine:
    def calculate(self, snapshot: FinancialSnapshot) -> QualityMetrics:
        invested_capital = None
        if snapshot.total_equity is not None and snapshot.total_debt is not None and snapshot.cash is not None:
            invested_capital = snapshot.total_equity + snapshot.total_debt - snapshot.cash

        nopat = snapshot.operating_income * 0.8 if snapshot.operating_income is not None else None
        history = snapshot.financial_history

        return QualityMetrics(
            roe=safe_divide(snapshot.net_income, snapshot.total_equity),
            roic=safe_divide(nopat, invested_capital),
            gross_margin=safe_divide(snapshot.gross_profit, snapshot.revenue),
            operating_margin=safe_divide(snapshot.operating_income, snapshot.revenue),
            net_margin=safe_divide(snapshot.net_income, snapshot.revenue),
            debt_to_equity=safe_divide(snapshot.total_debt, snapshot.total_equity),
            interest_coverage=safe_divide(snapshot.operating_income, snapshot.interest_expense),
            current_ratio=safe_divide(snapshot.current_assets, snapshot.current_liabilities),
            quick_ratio=safe_divide(
                (snapshot.current_assets - snapshot.inventory)
                if snapshot.current_assets is not None and snapshot.inventory is not None
                else None,
                snapshot.current_liabilities,
            ),
            fcf_margin=safe_divide(snapshot.free_cash_flow, snapshot.revenue),
            revenue_cagr_1y=self._cagr(history.get("revenue", []), 1),
            revenue_cagr_3y=self._cagr(history.get("revenue", []), 3),
            revenue_cagr_5y=self._cagr(history.get("revenue", []), 5),
            revenue_cagr_10y=self._cagr(history.get("revenue", []), 10),
            eps_cagr_1y=self._cagr(history.get("eps", []), 1),
            eps_cagr_3y=self._cagr(history.get("eps", []), 3),
            eps_cagr_5y=self._cagr(history.get("eps", []), 5),
            eps_cagr_10y=self._cagr(history.get("eps", []), 10),
            fcf_cagr_1y=self._cagr(history.get("free_cash_flow", []), 1),
            fcf_cagr_3y=self._cagr(history.get("free_cash_flow", []), 3),
            fcf_cagr_5y=self._cagr(history.get("free_cash_flow", []), 5),
            fcf_cagr_10y=self._cagr(history.get("free_cash_flow", []), 10),
            revenue_growth_3m=self._period_growth(history.get("quarterly_revenue", []), 1),
            revenue_growth_6m=self._period_growth(history.get("quarterly_revenue", []), 2),
            revenue_growth_9m=self._period_growth(history.get("quarterly_revenue", []), 3),
            eps_growth_3m=self._period_growth(history.get("quarterly_eps", []), 1),
            eps_growth_6m=self._period_growth(history.get("quarterly_eps", []), 2),
            eps_growth_9m=self._period_growth(history.get("quarterly_eps", []), 3),
        )

    @staticmethod
    def _cagr(values: list[float], years: int) -> float | None:
        if len(values) <= years or values[years] <= 0 or values[0] <= 0:
            return None
        return (values[0] / values[years]) ** (1 / years) - 1

    @staticmethod
    def _period_growth(values: list[float], periods: int) -> float | None:
        if len(values) <= periods or values[periods] == 0:
            return None
        return values[0] / values[periods] - 1


# =========================
# Decision / scoring
# =========================
@dataclass(frozen=True)
class Recommendation:
    label: str
    stars: str
    score: float | None


class DecisionEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.thresholds = config["scoring"]["recommendations"]

    def decide(self, score: float | None) -> Recommendation:
        if score is None:
            return Recommendation("Insufficient Data", "☆☆☆☆☆", None)
        if score >= self.thresholds["strong_buy"]:
            return Recommendation("Strong Buy", "★★★★★", score)
        if score >= self.thresholds["buy"]:
            return Recommendation("Buy", "★★★★☆", score)
        if score >= self.thresholds["hold"]:
            return Recommendation("Hold", "★★★☆☆", score)
        if score >= self.thresholds["reduce"]:
            return Recommendation("Reduce", "★★☆☆☆", score)
        return Recommendation("Avoid", "★☆☆☆☆", score)


class ScoringEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config
        
        
    @staticmethod #新增短期動能評估
    def short_growth_score(metrics: Any) -> float | None:
        """
        短期季度動能分數。

        看最新一季相對前 1、2、3 季的營收與 EPS 成長。
        不納入 Total Score，僅作為近期營運加速/減速的輔助觀察。
        """

        short_growth_metrics = {
            # EPS：60%
            "eps_growth_3m": 0.25,
            "eps_growth_6m": 0.20,
            "eps_growth_9m": 0.15,

            # Revenue：40%
            "revenue_growth_3m": 0.15,
            "revenue_growth_6m": 0.15,
            "revenue_growth_9m": 0.10,
        }

        full_score_growth = 0.30  # 成長 30% 以上才滿分

        score = 0.0
        total_weight = 0.0

        for field, weight in short_growth_metrics.items():
            value = getattr(metrics, field, None)

            if value is None:
                continue

            # 負成長與零成長均不給動能分；
            # 正成長依 0%~30% 線性給分；
            # 超過 30% 封頂，避免小基期造成極端值失真。
            normalized = min(
                max(value / full_score_growth, 0.0),
                1.0,
            )

            score += normalized * weight
            total_weight += weight

        if total_weight == 0:
            return None

        return round(score / total_weight * 100, 1)
    
    

    def quality_score(self, metrics: Any, fcf_yield: float | None) -> float | None:
        data = asdict(metrics)
        data["fcf_yield"] = fcf_yield

        total = 0.0
        available_weight = 0.0

        for name, rule in self.config["quality_score"]["metrics"].items():
            value = data.get(name)
            if value is None:
                continue

            weight = rule["weight"]
            excellent = rule["excellent"]
            good = rule["good"]
            lower = rule.get("lower_is_better", False)

            if lower:
                points = 1.0 if value <= excellent else 0.6 if value <= good else 0.2
            else:
                points = 1.0 if value >= excellent else 0.6 if value >= good else 0.2

            total += weight * points
            available_weight += weight

        return 100 * total / available_weight if available_weight else None
    
    
    
    
    @staticmethod #新增訊號解讀
    def _growth_sign(
        value: float | None,
        neutral_threshold: float = 0.0,
    ) -> str:
        """
        將成長率轉為正向 / 負向 / 持平。

        neutral_threshold 預設為 0：
        > 0  為正向
        < 0  為負向
        = 0  為持平

        日後若想忽略小幅波動，可改為 0.02，
        代表 -2% ~ +2% 視為持平。
        """
        if value is None:
            return "missing"

        if value > neutral_threshold:
            return "positive"

        if value < -neutral_threshold:
            return "negative"

        return "neutral"


    @classmethod
    def growth_momentum_direction(
        cls,
        growth_3m: float | None,
        growth_6m: float | None,
        growth_9m: float | None,
        neutral_threshold: float = 0.0,
    ) -> str:
        """
        依 3M / 6M / 9M 成長率判讀短期動能方向。

        3M：最新季相較前一季
        6M：最新季相較前兩季
        9M：最新季相較前三季
        """

        if (
            growth_3m is None
            or growth_6m is None
            or growth_9m is None
        ):
            return "資料不足"

        sign_3m = cls._growth_sign(
            growth_3m,
            neutral_threshold,
        )
        sign_6m = cls._growth_sign(
            growth_6m,
            neutral_threshold,
        )
        sign_9m = cls._growth_sign(
            growth_9m,
            neutral_threshold,
        )

        # 任何一項接近 0 時，視為盤整，避免過度判讀
        if "neutral" in {sign_3m, sign_6m, sign_9m}:
            if sign_3m == sign_6m == sign_9m == "neutral":
                return "持平盤整"
            return "正負交錯／盤整"

        # -------------------------------------------------
        # 三者皆為正
        # -------------------------------------------------
        if (
            sign_3m == "positive"
            and sign_6m == "positive"
            and sign_9m == "positive"
        ):
            if growth_3m > growth_6m > growth_9m:
                return "正向且加速"

            if growth_3m < growth_6m < growth_9m:
                return "正向但減速"

            return "正向無趨勢"

        # -------------------------------------------------
        # 三者皆為負
        # 注意：負值越接近 0，代表衰退正在改善
        # -------------------------------------------------
        if (
            sign_3m == "negative"
            and sign_6m == "negative"
            and sign_9m == "negative"
        ):
            # 例如：-5% > -10% > -20%
            # 代表衰退幅度縮小，負向改善
            if growth_3m > growth_6m > growth_9m:
                return "負向強改善"

            # 例如：-20% < -10% < -5%
            # 代表最新季衰退最嚴重，負向惡化
            if growth_3m < growth_6m < growth_9m:
                return "負向強惡化"

            return "負向無趨勢"

        # -------------------------------------------------
        # 負轉正：最新季度已轉正
        # -------------------------------------------------
        if (
            sign_3m == "positive"
            and sign_6m == "positive"
            and sign_9m == "negative"
        ):
            return "負向轉強正向"

        if (
            sign_3m == "positive"
            and sign_6m == "negative"
            and sign_9m == "negative"
        ):
            return "負向轉正向"

        # 正、負、正：趨勢不穩
        if (
            sign_3m == "positive"
            and sign_6m == "negative"
            and sign_9m == "positive"
        ):
            return "正向波動"

        # -------------------------------------------------
        # 正轉負：最新季度已轉負
        # -------------------------------------------------
        if (
            sign_3m == "negative"
            and sign_6m == "negative"
            and sign_9m == "positive"
        ):
            return "正向轉強負向"

        if (
            sign_3m == "negative"
            and sign_6m == "positive"
            and sign_9m == "positive"
        ):
            return "正向轉負向"

        # 負、正、負：趨勢不穩
        if (
            sign_3m == "negative"
            and sign_6m == "positive"
            and sign_9m == "negative"
        ):
            return "負向波動"

        return "趨勢不明"


    @classmethod
    def short_growth_direction(
        cls,
        metrics: Any,
        neutral_threshold: float = 0.0,
    ) -> dict[str, str]:
        """
        分別判讀 EPS、營收動能，
        並提供一個簡化的綜合趨勢結果。

        不影響 Total Score，僅作觀察用途。
        """

        eps_direction = cls.growth_momentum_direction(
            metrics.eps_growth_3m,
            metrics.eps_growth_6m,
            metrics.eps_growth_9m,
            neutral_threshold,
        )

        revenue_direction = cls.growth_momentum_direction(
            metrics.revenue_growth_3m,
            metrics.revenue_growth_6m,
            metrics.revenue_growth_9m,
            neutral_threshold,
        )

        # 綜合判讀邏輯
        if eps_direction == "資料不足" and revenue_direction == "資料不足":
            overall_direction = "資料不足"

        elif eps_direction == revenue_direction:
            overall_direction = eps_direction

        elif (
            "正向" in eps_direction
            and "正向" in revenue_direction
        ):
            overall_direction = "正向但營收／EPS趨勢不同"

        elif (
            "負向" in eps_direction
            and "負向" in revenue_direction
        ):
            overall_direction = "負向但營收／EPS趨勢不同"

        elif (
            "負向轉正向" in eps_direction
            or "負向轉強正向" in eps_direction
        ) and "正向" in revenue_direction:
            overall_direction = "獲利轉強，營收正向"

        elif (
            "負向轉正向" in revenue_direction
            or "負向轉強正向" in revenue_direction
        ) and "正向" in eps_direction:
            overall_direction = "營收轉強，獲利正向"

        elif (
            "正向轉負向" in eps_direction
            or "正向轉強負向" in eps_direction
        ):
            overall_direction = "獲利趨勢轉弱"

        elif (
            "正向轉負向" in revenue_direction
            or "正向轉強負向" in revenue_direction
        ):
            overall_direction = "營收趨勢轉弱"

        else:
            overall_direction = "營收／EPS趨勢分歧"

        return {
            "eps_direction": eps_direction,
            "revenue_direction": revenue_direction,
            "overall_direction": overall_direction,
        }

    @staticmethod
    def valuation_score(pe_percentile: float | None) -> float | None:
        return None if pe_percentile is None else max(0.0, min(100.0, 100 - pe_percentile))

    def growth_score(self, metrics: Any) -> float | None:
        rules = self.config.get("growth_score", {}).get("metrics", {})
        if not rules:
            return None

        total = 0.0
        available_weight = 0.0
        for name, rule in rules.items():
            value = getattr(metrics, name, None)
            if value is None:
                continue

            weight = rule["weight"]
            excellent = rule["excellent"]
            good = rule["good"]
            lower = rule.get("lower_is_better", False)
            if lower:
                points = 1.0 if value <= excellent else 0.6 if value <= good else 0.2
            else:
                points = 1.0 if value >= excellent else 0.6 if value >= good else 0.2
            total += weight * points
            available_weight += weight

        return 100 * total / available_weight if available_weight else None
    
        #買賣時機分數  
    def technical_score(
        self,
        technical,
        price
    ):

        score = 0
        total_weight = 0


        # ==========================
        # 1. RSI (35%)
        # 越低越有買點價值
        # ==========================

        if technical.rsi is not None:

            total_weight += 35

            rsi = technical.rsi

            if rsi < 30:
                score += 35

            elif rsi < 40:
                score += 30

            elif rsi <=50:
                score += 25

            elif rsi <=60:
                score += 15

            elif rsi <=70:
                score += 5

            else:
                score += 0



        # ==========================
        # 2. 支撐距離 (35%)
        # 越靠近支撐越好
        # ==========================

        if technical.support is not None:

            total_weight +=35

            distance = (
                price - technical.support
            ) / price


            if distance <=0.02:
                score +=35

            elif distance <=0.05:
                score +=30

            elif distance <=0.10:
                score +=20

            else:
                score +=5



        # ==========================
        # 3. 壓力距離 (20%)
        # 避免追高
        # ==========================

        if technical.resistance is not None:

            total_weight +=20

            distance = (
                technical.resistance-price
            ) / price


            # 還有很大上漲空間
            if distance >=0.15:
                score +=20

            elif distance >=0.08:
                score +=15

            elif distance >=0.03:
                score +=8

            else:
                score +=0



        # ==========================
        # 4. 均線趨勢 (10%)
        # 趨勢只輔助
        # ==========================

        if (
            technical.short_ma is not None
            and technical.long_ma is not None
        ):

            total_weight +=10


            if technical.short_ma > technical.long_ma:
                score +=10

            elif technical.short_ma >= technical.long_ma*0.97:
                score +=5

            else:
                score +=0



        return (
            None
            if total_weight==0
            else score / total_weight *100
        )
    
    #籌碼面分數
    def flow_score(
        self,
        flow_signal: str | None,
        volume_ratio: float | None = None,
        return_20d: float | None = None,
    ) -> float | None:

        if flow_signal in {"Unavailable", "Insufficient Data"}:
            flow_signal = None


        score = 0
        total = 0


        # =====================
        # 資金訊號
        # =====================

        if flow_signal is not None:

            total += 40

            mapping = {

                "Strong Buy": 40,
                "Buy": 30,
                "Neutral": 20,
                "Sell": 10,
                "Sell/Reduce": 10,
                "Strong Sell": 0,

                # 中文防呆
                "強買":40,
                "買進":30,
                "中性":20,
                "賣出":10,
                "強賣":0,
            }

            score += mapping.get(
                flow_signal,
                20
            )


        # =====================
        # 成交量
        # =====================

        if volume_ratio is not None:

            total += 30


            if volume_ratio >= 1.5:
                score += 30

            elif volume_ratio >= 1.2:
                score += 20

            elif volume_ratio >= 1:
                score += 10

            else:
                score += 5



        # =====================
        # 20日報酬
        # =====================

        if return_20d is not None:

            total += 30


            if return_20d >= 0.10:
                score += 30

            elif return_20d >= 0.05:
                score += 20

            elif return_20d >= 0:
                score += 10

            else:
                score += 5



        # =====================
        # 防呆
        # =====================

        if total == 0:
            return None


        return score / total * 100
    

    def total_score(
        self,
        valuation: float | None,
        quality: float | None,
        growth: float | None,
        buffett: float | None,
    ) -> float | None:
        pairs = [
            ("valuation", valuation),
            ("quality", quality),
            ("growth", growth),
            ("buffett", buffett),
        ]
        weights = self.config["scoring"]["weights"]
        usable = [(weights[name], value) for name, value in pairs if value is not None]
        return None if not usable else sum(weight * value for weight, value in usable) / sum(weight for weight, _ in usable)    

class BuffettChecklist:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["buffett"]

    def evaluate(self, metrics: Any, snapshot: Any) -> dict[str, bool | None]:
        return {
            "roic": self._compare(metrics.roic, self.rules.get("roic_min"), True),
            "roe": self._compare(metrics.roe, self.rules.get("roe_min"), True),
            "debt_to_equity": self._compare(metrics.debt_to_equity, self.rules.get("debt_to_equity_max"), False),
            "fcf_positive": None if snapshot.free_cash_flow is None else snapshot.free_cash_flow > 0,
            "eps_growth_positive": None if metrics.eps_cagr_3y is None else metrics.eps_cagr_3y > 0,
            "operating_margin_positive": None if metrics.operating_margin is None else metrics.operating_margin > 0,
            "share_count_not_increasing": None,
        }

    @staticmethod
    def _compare(value: float | None, threshold: float | None, greater: bool) -> bool | None:
        if value is None or threshold is None:
            return None
        return value >= threshold if greater else value <= threshold

    @staticmethod
    def score(checks: dict[str, bool | None]) -> float | None:
        available = [value for value in checks.values() if value is not None]
        return None if not available else 100 * sum(available) / len(available)


# =========================
# Valuation
# =========================

@dataclass
class ValuationMetrics:
    pe: float | None = None
    forward_pe: float | None = None          # market / Yahoo forward PE
    pb: float | None = None
    peg: float | None = None
    fcf_yield: float | None = None
    ev_ebit: float | None = None
    ev_ebitda: float | None = None
    dcf_value_per_share: float | None = None
    historical_pe: dict[str, float | None] | None = None
    historical_pb: dict[str, float | None] | None = None
    # ========= Historical PE/PB distribution =========
    historical_pe_stats: dict[str, float | None] | None = None
    historical_pb_stats: dict[str, float | None] | None = None
    historical_pe_percentile: float | None = None
    historical_pb_percentile: float | None = None    
    nowcast_pe_percentile: float | None = None   
    pe_percentile_5y: float | None = None
    pb_percentile_5y: float | None = None
    pe_sample_count: int = 0
    pe_history_years: float | None = None
        
        
class ValuationEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config["valuation"]

    def calculate(
        self,
        snapshot: FinancialSnapshot,
        price_history: pd.DataFrame,
        historical_fundamentals: pd.DataFrame | None = None,
    ) -> ValuationMetrics:
        pe = safe_divide(snapshot.current_price, snapshot.eps)
        forward_pe = safe_divide(snapshot.current_price, snapshot.forward_eps)  # Yahoo / market forward PE
        pb = safe_divide(snapshot.current_price, snapshot.book_value_per_share)
        growth = self._cagr(snapshot.financial_history.get("eps", []), 3)
        historical_ratios = self._point_in_time_ratios(price_history, historical_fundamentals) #歷史估值百分位 (年)
        historical_pe_stats = self._historical_ratio_statistics(historical_ratios, "pe", )  #歷史PE估值百分位 (季)
        historical_pb_stats = self._historical_ratio_statistics(historical_ratios, "pb", )  #歷史PB估值百分位 (季)
        
        pe_series = historical_ratios["pe"].dropna().tolist() if not historical_ratios.empty else []
        pb_series = historical_ratios["pb"].dropna().tolist() if not historical_ratios.empty else []
        nowcast_pe = snapshot.pe_ttm_nowcast
        nowcast_pe_percentile = percentile_rank(nowcast_pe, pe_series, )
        pe_dates = historical_ratios.loc[historical_ratios["pe"].notna(), "date"] if not historical_ratios.empty else pd.Series(dtype="datetime64[ns]")
        pe_history_years = (pe_dates.max() - pe_dates.min()).days / 365.25 if len(pe_dates) > 1 else None
        historical_pe = self._windowed_averages(historical_ratios, "pe")
        historical_pb = self._windowed_averages(historical_ratios, "pb")        
        # 新增：歷史PE序列
        historical_pe_values = (
            historical_ratios["pe"]
            .dropna()
            .tolist()
        )
        
        current_pe_percentile = None
        if (snapshot.current_price
            and snapshot.ttm_eps
            and snapshot.ttm_eps > 0
            and historical_pe_values
        ):
            current_pe = (
                snapshot.current_price /
                snapshot.ttm_eps
            )
            current_pe_percentile = (
                sum(
                    1 
                    for pe in historical_pe_values
                    if pe <= current_pe
                )
                /
                len(historical_pe_values)
            )

        return ValuationMetrics(
            pe=pe,
            forward_pe=forward_pe,
            pb=pb,
            peg=safe_divide(pe, growth) if growth and growth > 0 else None,
            fcf_yield=safe_divide(snapshot.free_cash_flow, snapshot.market_cap),
            ev_ebit=safe_divide(snapshot.enterprise_value, snapshot.operating_income),
            dcf_value_per_share=self._dcf(snapshot),
            historical_pe=historical_pe,
            historical_pb=historical_pb,
            historical_pe_stats=historical_pe_stats,
            historical_pb_stats=historical_pb_stats,
            historical_pe_percentile=current_pe_percentile,
            nowcast_pe_percentile=nowcast_pe_percentile,
            pe_percentile_5y=percentile_rank(pe, pe_series),
            pb_percentile_5y=percentile_rank(pb, pb_series),
            pe_sample_count=len(pe_series),
            pe_history_years=pe_history_years,
        )

    
    def _point_in_time_ratios(self, history: pd.DataFrame, fundamentals: pd.DataFrame | None) -> pd.DataFrame:
        if history is None or history.empty or fundamentals is None or fundamentals.empty or "Close" not in history:
            return pd.DataFrame(columns=["date", "pe", "pb"])

        prices = history[["Close"]].copy().sort_index()
        prices.index = pd.to_datetime(prices.index, utc=True).tz_convert(None)
        monthly_prices = prices["Close"].resample("ME").last().dropna().rename("close").reset_index()
        monthly_prices.columns = ["date", "close"]

        observations = fundamentals.copy()
        lag_days = self.config.get("earnings_availability_lag_days", 45)
        observations["effective_date"] = pd.to_datetime(observations["report_date"], utc=True).dt.tz_convert(None) + pd.DateOffset(days=lag_days)
        observations = observations.sort_values(["effective_date", "report_date"]).drop_duplicates("effective_date", keep="last")

        merged = pd.merge_asof(
            monthly_prices.sort_values("date"),
            observations[["effective_date", "ttm_eps", "bvps"]],
            left_on="date",
            right_on="effective_date",
            direction="backward",
        )
        merged["pe"] = [safe_divide(close, eps) if pd.notna(eps) and eps > 0 else None for close, eps in zip(merged["close"], merged["ttm_eps"])]
        merged["pb"] = [safe_divide(close, bvps) if pd.notna(bvps) and bvps > 0 else None for close, bvps in zip(merged["close"], merged["bvps"])]
        return merged[["date", "pe", "pb"]].dropna(how="all", subset=["pe", "pb"]).reset_index(drop=True)

    def _windowed_averages(self, ratios: pd.DataFrame, column: str) -> dict[str, float | None]:
        result: dict[str, float | None] = {}

        if ratios.empty:
            for year in self.config.get("historical_windows_years", []):
                result[f"{year}y"] = None
            for month in self.config.get("historical_window_months", []):
                result[f"{month}m"] = None
            return result

        latest = ratios["date"].max()

        for years in self.config.get("historical_windows_years", []):
            values = ratios.loc[ratios["date"] >= latest - pd.DateOffset(years=years), column].dropna()
            result[f"{years}y"] = float(values.mean()) if not values.empty else None

        for months in self.config.get("historical_window_months", []):
            values = ratios.loc[ratios["date"] >= latest - pd.DateOffset(months=months), column].dropna()
            result[f"{months}m"] = float(values.mean()) if not values.empty else None

        return result
        
    def _historical_ratio_statistics(
        self,
        ratios: pd.DataFrame,
        column: str,
    ) -> dict[str, float | None] | None:
        """
        對歷史 PE 或 PB 序列計算分位數統計。

        column:
          - "pe"
          - "pb"
        """
        if ratios is None or ratios.empty or column not in ratios.columns:
            return None

        values = ratios[column].dropna().astype(float)

        # 月頻資料至少 8 筆才建立價格帶
        if len(values) < 8:
            return None

        return {
            "mean": float(values.mean()),
            "p05": float(values.quantile(0.05)),
            "p25": float(values.quantile(0.25)),
            "p50": float(values.quantile(0.50)),
            "p75": float(values.quantile(0.75)),
            "p95": float(values.quantile(0.95)),
            "min": float(values.min()),
            "max": float(values.max()),
            "count": float(len(values)),
        }    
    
       
    def _dcf(self, snapshot: FinancialSnapshot) -> float | None:
        fcf = snapshot.free_cash_flow
        shares = snapshot.shares_outstanding
        if fcf is None or shares in (None, 0):
            return None

        assumptions = self.config["dcf"]
        rate = assumptions["discount_rate"]
        terminal = assumptions["terminal_growth_rate"]
        years = assumptions["forecast_years"]

        if rate <= terminal:
            return None

        projected = sum(fcf / ((1 + rate) ** year) for year in range(1, years + 1))
        terminal_value = fcf * (1 + terminal) / (rate - terminal) / ((1 + rate) ** years)

        return (projected + terminal_value) / shares

    @staticmethod
    def _cagr(values: list[float], years: int) -> float | None:
        if len(values) <= years or values[years] <= 0 or values[0] <= 0:
            return None
        return (values[0] / values[years]) ** (1 / years) - 1


# =========================
# Trading / profile
# =========================       
@dataclass
class PriceTargets:

    # =========================
    # Model valuation
    # =========================
    model_buy_price: float | None
    model_fair_price: float | None
    model_sell_price: float | None

    # =========================
    # Historical PE valuation
    # =========================
    
    historical_P05_price: float | None = None
    historical_buy_price: float | None = None
    historical_fair_price: float | None = None
    historical_sell_price: float | None = None
    historical_P95_price: float | None = None    

    upside_to_fair: float | None = None

    primary_metric: str = ""
    basis: str = ""



@dataclass
class TechnicalSignal:
    signal: str
    rsi: float | None
    short_ma: float | None
    long_ma: float | None
    support: float | None
    resistance: float | None
        
@dataclass
class USFlowProxySignal:
    signal: str
    volume_ratio: float | None
    short_ma: float | None
    long_ma: float | None
    return_20d: float | None
    reason: str

@dataclass
class ChipSignal:
    signal: str
    stop_loss: float | None
    take_profit: float | None
    reason: str

class ProfileResolver:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config

    def classify(self, ticker: str, asset_type: str | None = None) -> Classification:
        override = self.config.get("ticker_classification", {}).get(ticker)
        if override:
            return Classification(**override)

        market = "TW" if ticker.endswith((".TW", ".TWO")) else "US"
        sector = "traditional" if asset_type in {"ETF", "MUTUALFUND"} else "technology"
        return Classification(market=market, sector=sector)

    def rules(self, classification: Classification, ticker: str | None = None) -> dict[str, Any]:
        market = self.config["market_profiles"][classification.market]
        sector = self.config["sector_profiles"][classification.sector]
        override = self.config.get("ticker_rule_overrides", {}).get(ticker or "", {})
        return {
            "valuation": market["valuation"] | sector["valuation"] | override.get("valuation", {}),
            "trading": market["trading"] | override.get("trading", {}),
            "quality": sector["quality"] | override.get("quality", {}),
        }


class PriceTargetEngine:
    """
    依股票估值主指標產生價格帶：

    PE 型股票：
        歷史 PE 分位數 × TTM EPS

    PB 型股票：
        歷史 PB 分位數 × BVPS
    """

    @staticmethod
    def _normalized_base(
        snapshot: FinancialSnapshot,
        metric: str,
    ) -> float | None:
        """
        當沒有 Forward EPS Model 時，可作為 PE 型估值的備援基礎。
        PB 型股票直接使用最新 BVPS。
        """
        if metric == "pb":
            return snapshot.book_value_per_share

        # PE 型：優先使用歷史年度 EPS 中位數，
        # 沒有資料則使用目前 EPS。
        values = sorted(
            value
            for value in snapshot.financial_history.get("eps", [])
            if value is not None and value > 0
        )

        if not values:
            return snapshot.ttm_eps or snapshot.eps

        midpoint = len(values) // 2

        if len(values) % 2:
            return float(values[midpoint])

        return float(
            (values[midpoint - 1] + values[midpoint]) / 2
        )

    @staticmethod
    def _fair_multiple(
        valuation_metrics: ValuationMetrics,
        metric: str,
        cheap: float,
        expensive: float,
    ) -> float:
        """
        若有歷史平均估值，以歷史 3Y / 5Y / 1Y 平均為優先；
        否則使用 YAML cheap / expensive 的中間值。
        """
        historical = (
            valuation_metrics.historical_pe
            if metric == "pe"
            else valuation_metrics.historical_pb
        )

        if historical:
            for window in ("3y", "5y", "1y"):
                value = historical.get(window)

                if value is not None and value > 0:
                    return max(cheap, min(expensive, value))

        return (cheap + expensive) / 2

    @staticmethod
    def _get_historical_stats(
        valuation_metrics: ValuationMetrics,
        metric: str,
    ) -> dict[str, float | None] | None:
        """依估值主指標取得 PE 或 PB 的歷史分位數統計。"""
        if metric == "pe":
            return valuation_metrics.historical_pe_stats

        if metric == "pb":
            return valuation_metrics.historical_pb_stats

        return None

    @staticmethod
    def _get_trailing_base(
        snapshot: FinancialSnapshot,
        metric: str,
    ) -> float | None:
        """
        取得歷史價格帶的每股基礎：

        PE → TTM EPS，若無則 fallback EPS
        PB → 最新 BVPS
        """
        if metric == "pe":
            base = snapshot.ttm_eps or snapshot.eps
        elif metric == "pb":
            base = snapshot.book_value_per_share
        else:
            base = None

        if base is None or base <= 0:
            return None

        return float(base)

    def calculate(
        self,
        snapshot: FinancialSnapshot,
        rules: dict[str, Any],
        valuation_metrics: ValuationMetrics,
    ) -> PriceTargets:
        valuation = rules["valuation"]

        metric = valuation["primary_metric"].lower()
        cheap = valuation[f"{metric}_cheap"]
        expensive = valuation[f"{metric}_expensive"]

        # 依 metric 自動選 PE / PB 歷史統計
        historical_stats = self._get_historical_stats(
            valuation_metrics,
            metric,
        )

        # 歷史價格帶使用 TTM EPS 或 BVPS
        trailing_base = self._get_trailing_base(
            snapshot,
            metric,
        )

        # 模型價格帶的 PE 基礎：
        # 優先使用 FinMind 月營收推估的即時 EPS，
        # 若無法計算，再退回正式 TTM EPS。
        if (
            metric == "pe"
            and snapshot.eps_ttm_nowcast is not None
            and snapshot.eps_ttm_nowcast > 0
        ):
            fair_base = snapshot.eps_ttm_nowcast
        else:
            fair_base = trailing_base

            
        # 若主要基礎資料缺失，再用中位數 EPS/BVPS 當 fallback
        if fair_base is None:
            fair_base = self._normalized_base(
                snapshot,
                metric,
            )

        # -------------------------------------------------
        # 決定模型估值倍數
        # 優先使用歷史 P25 / P50 / P75
        # -------------------------------------------------
        if (
            historical_stats is not None
            and historical_stats.get("p25") is not None
            and historical_stats.get("p50") is not None
            and historical_stats.get("p75") is not None
        ):
            cheap_multiple = historical_stats["p25"]
            fair_multiple = historical_stats["p50"]
            expensive_multiple = historical_stats["p75"]
        else:
            # 無足夠歷史資料時，退回 YAML 規則
            cheap_multiple = cheap
            fair_multiple = self._fair_multiple(
                valuation_metrics,
                metric,
                cheap,
                expensive,
            )
            expensive_multiple = expensive

        # -------------------------------------------------
        # 歷史分位數價格帶：P05 / P25 / P50 / P75 / P95
        # PE：TTM EPS × 歷史 PE 分位數
        # PB：BVPS × 歷史 PB 分位數
        # -------------------------------------------------
        historical_P05_price = None
        historical_buy_price = None
        historical_fair_price = None
        historical_sell_price = None
        historical_P95_price = None

        required_percentiles = ["p05", "p25", "p50", "p75", "p95"]

        if (
            historical_stats is not None
            and trailing_base is not None
            and all(
                historical_stats.get(key) is not None
                for key in required_percentiles
            )
        ):
            historical_P05_price = (
                trailing_base * historical_stats["p05"]
            )
            historical_buy_price = (
                trailing_base * historical_stats["p25"]
            )
            historical_fair_price = (
                trailing_base * historical_stats["p50"]
            )
            historical_sell_price = (
                trailing_base * historical_stats["p75"]
            )
            historical_P95_price = (
                trailing_base * historical_stats["p95"]
            )

        # -------------------------------------------------
        # 模型價格帶
        # -------------------------------------------------
        model_buy_price = (
            None
            if fair_base is None
            else fair_base * cheap_multiple
        )

        model_fair_price = (
            None
            if fair_base is None
            else fair_base * fair_multiple
        )

        model_sell_price = (
            None
            if fair_base is None
            else fair_base * expensive_multiple
        )

        upside_to_fair = (
            safe_divide(
                model_fair_price - snapshot.current_price,
                snapshot.current_price,
            )
            if (
                model_fair_price is not None
                and snapshot.current_price not in (None, 0)
            )
            else None
        )

        # 說明文字依 PE/PB 改變
        if metric == "pe":
            historical_basis = "historical PE percentile × TTM EPS"
            model_basis = (
                "model uses forward EPS when available"
            )
        elif metric == "pb":
            historical_basis = "historical PB percentile × BVPS"
            model_basis = "model uses latest BVPS"
        else:
            historical_basis = "historical valuation percentile"
            model_basis = "model valuation"

        return PriceTargets(
            model_buy_price=model_buy_price,
            model_fair_price=model_fair_price,
            model_sell_price=model_sell_price,

            historical_P05_price=historical_P05_price,
            historical_buy_price=historical_buy_price,
            historical_fair_price=historical_fair_price,
            historical_sell_price=historical_sell_price,
            historical_P95_price=historical_P95_price,

            upside_to_fair=upside_to_fair,
            primary_metric=metric.upper(),
            basis=(
                f"{metric.upper()} valuation; "
                f"{model_basis}; "
                f"historical uses {historical_basis}"
            ),
        )
    

class TechnicalAnalysisEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["technical"]

    def analyze(self, prices: pd.DataFrame) -> TechnicalSignal:
        if prices is None or prices.empty or "Close" not in prices:
            return TechnicalSignal("Unavailable", None, None, None, None, None)

        close = prices["Close"].dropna()
        minimum = max(self.rules["long_ma_days"], self.rules["rsi_days"] + 1)

        if len(close) < minimum:
            return TechnicalSignal("Insufficient Data", None, None, None, None, None)

        short_ma = float(close.tail(self.rules["short_ma_days"]).mean())
        long_ma = float(close.tail(self.rules["long_ma_days"]).mean())

        delta = close.diff()
        gains = delta.clip(lower=0)
        losses = -delta.clip(upper=0)

        avg_gain = gains.ewm(alpha=1 / self.rules["rsi_days"], adjust=False).mean().iloc[-1]
        avg_loss = losses.ewm(alpha=1 / self.rules["rsi_days"], adjust=False).mean().iloc[-1]

        rsi = 100.0 if avg_loss == 0 else float(100 - 100 / (1 + avg_gain / avg_loss))
        support = float(close.tail(self.rules["support_lookback_days"]).min())
        resistance = float(close.tail(self.rules["resistance_lookback_days"]).max())
        latest = float(close.iloc[-1])

        if short_ma > long_ma and rsi <= self.rules["rsi_oversold"]:
            signal = "Buy Setup"
        elif short_ma < long_ma or rsi >= self.rules["rsi_overbought"]:
            signal = "Sell/Reduce Setup"
        elif latest <= support * 1.02:
            signal = "Watch Buy Zone"
        else:
            signal = "Hold/Wait"

        return TechnicalSignal(signal, rsi, short_ma, long_ma, support, resistance)
    
class USFlowProxyEngine:
    """
    US large-money / institutional-flow proxy using price and volume.
    """
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config.get("us_flow_proxy", {
            "short_ma_days": 20,
            "long_ma_days": 60,
            "volume_short_days": 5,
            "volume_long_days": 60,
            "volume_ratio_threshold": 1.3,
            "return_lookback_days": 20,
        })

    def analyze(self, prices: pd.DataFrame) -> USFlowProxySignal:
        if prices is None or prices.empty or "Close" not in prices or "Volume" not in prices:
            return USFlowProxySignal(
                signal="Unavailable",
                volume_ratio=None,
                short_ma=None,
                long_ma=None,
                return_20d=None,
                reason="Price/volume history unavailable",
            )

        close = prices["Close"].dropna()
        volume = prices["Volume"].dropna()

        minimum = max(
            self.rules["long_ma_days"],
            self.rules["volume_long_days"],
            self.rules["return_lookback_days"] + 1,
        )

        if len(close) < minimum or len(volume) < minimum:
            return USFlowProxySignal(
                signal="Insufficient Data",
                volume_ratio=None,
                short_ma=None,
                long_ma=None,
                return_20d=None,
                reason="Not enough price/volume history",
            )

        short_ma = float(close.tail(self.rules["short_ma_days"]).mean())
        long_ma = float(close.tail(self.rules["long_ma_days"]).mean())
        latest = float(close.iloc[-1])

        short_vol = float(volume.tail(self.rules["volume_short_days"]).mean())
        long_vol = float(volume.tail(self.rules["volume_long_days"]).mean())
        volume_ratio = safe_divide(short_vol, long_vol)

        lookback = self.rules["return_lookback_days"]
        past_price = float(close.iloc[-lookback - 1])
        return_20d = safe_divide(latest - past_price, past_price)

        bullish = (
            latest > short_ma > long_ma and
            (volume_ratio is not None and volume_ratio >= self.rules["volume_ratio_threshold"]) and
            (return_20d is not None and return_20d > 0)
        )

        bearish = (
            latest < short_ma < long_ma and
            (volume_ratio is not None and volume_ratio >= self.rules["volume_ratio_threshold"]) and
            (return_20d is not None and return_20d < 0)
        )

        vol_text = f"{volume_ratio:.2f}" if volume_ratio is not None else "NA"
        ret_text = f"{return_20d:.2%}" if return_20d is not None else "NA"

        if bullish:
            return USFlowProxySignal(
                signal="Buy",
                volume_ratio=volume_ratio,
                short_ma=short_ma,
                long_ma=long_ma,
                return_20d=return_20d,
                reason=f"Uptrend with expanding volume (vol_ratio={vol_text}, return_20d={ret_text})",
            )

        if bearish:
            return USFlowProxySignal(
                signal="Sell/Reduce",
                volume_ratio=volume_ratio,
                short_ma=short_ma,
                long_ma=long_ma,
                return_20d=return_20d,
                reason=f"Downtrend with expanding volume (vol_ratio={vol_text}, return_20d={ret_text})",
            )

        return USFlowProxySignal(
            signal="Hold",
            volume_ratio=volume_ratio,
            short_ma=short_ma,
            long_ma=long_ma,
            return_20d=return_20d,
            reason=f"Mixed/neutral flow proxy (vol_ratio={vol_text}, return_20d={ret_text})",
        )


class ChipAnalysisEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["chip_analysis"]

    def analyze(self, chip_data: pd.DataFrame, entry_price: float | None) -> ChipSignal:
        stop = entry_price * (1 - self.rules["stop_loss_from_entry_pct"]) if entry_price else None
        profit = entry_price * (1 + self.rules["take_profit_from_entry_pct"]) if entry_price else None

        if chip_data is None or chip_data.empty or "institutional_net_buy" not in chip_data:
            return ChipSignal("Unavailable", stop, profit, "Provider has no institutional-flow data")

        flow = chip_data["institutional_net_buy"].tail(self.rules["institutional_buying_days"]).dropna()

        if len(flow) < self.rules["institutional_buying_days"]:
            return ChipSignal("Insufficient Data", stop, profit, "Not enough recent institutional-flow observations")

        signal = "Buy" if (flow > 0).all() else "Sell/Reduce" if (flow < 0).all() else "Hold"
        return ChipSignal(signal, stop, profit, "Recent institutional net-buying trend")




# =========================
# Report / translations
# =========================
ZH_TW_COLUMNS = {
    "Ticker": "代號",
    "Source": "資料來源",
    "Market": "市場",
    "Sector": "產業",
    "Price": "現價",
    "PE": "本益比",
    "TTM EPS": "近年EPS",
    "EPS TTM Nowcast": "即時EPS",   
    "PE TTM Nowcast": "即時本益比",   
    "Forward EPS": "預估EPS",   
    "Forward PE": "預估本益比",    
    "PB": "股價淨值比",
    "ROIC": "投入資本報酬率",
    "ROE": "股東權益報酬率",
    "FCF Yield": "自由現金流殖利率",
    "Revenue Growth 3M": "營收近三月成長率",
    "Revenue Growth 6M": "營收近六月成長率",
    "EPS Growth 3M": "EPS 近三月成長率",
    "EPS Growth 6M": "EPS 近六月成長率",
    "EPS CAGR 3Y": "EPS 三年複合成長率",
    "Short Growth Score": "短期成長動能",
    "EPS Momentum Trend": "EPS短期趨勢",
    "Revenue Momentum Trend": "營收短期趨勢",
    "Short Growth Trend": "短期動能趨勢",
    "Valuation Percentile": "歷史估值百分位",
    "Valuation Sample Count": "歷史估值樣本數",
    "Valuation History Years": "歷史估值可用年數",
    "Growth Score": "成長分數",
    "Quality Score": "品質分數",
    "Buffett Score": "巴菲特檢核分數",
    "Fundamental Score": "基本面分數",
    "Recommendation": "投資建議",
    "Stars": "星等",
    "Model Buy Price": "模型預估便宜價",
    "Model Fair Price": "模型預估合理價",
    "Model Sell Price": "模型預估昂貴價",
    "Historical P05 Price": "歷史估值05%價格",
    "Historical Buy Price": "歷史估值25%價格",
    "Historical Fair Price": "歷史估值50%價格",
    "Historical Sell Price": "歷史估值75%價格",
    "Historical P95 Price": "歷史估值95%價格",
    "Nowcast PE Percentile":"即時歷史估值位階",
    "Fair Value Upside": "合理價上行空間",
    "Target Metric": "目標價依據",
    "Valuation Basis": "估值模型說明",
    "Technical Signal": "技術面訊號",
    "Technical Score": "技術面分數",
    "RSI": "相對強弱指標 RSI",
    "Support": "支撐價",
    "Resistance": "壓力價",
    "Flow Signal": "資金流訊號",
    "Flow Score": "資金流分數",
    "Flow Note": "資金流說明",
    "Error": "錯誤",
}

ZH_TW_VALUES = {
    "technology": "科技",
    "financial": "金融",
    "traditional": "傳產",
    "cyclical": "景氣循環",
    "Strong Buy": "強力買進",
    "Buy": "買進",
    "Hold": "持有",
    "Reduce": "減碼",
    "Avoid": "避開",
    "Insufficient Data": "資料不足",
    "Data Error": "資料錯誤",
    "Buy Setup": "買進訊號形成",
    "Sell/Reduce Setup": "賣出／減碼訊號",
    "Watch Buy Zone": "觀察買點區",
    "Hold/Wait": "持有／等待",
    "Unavailable": "暫無資料",
    "Sell/Reduce": "賣出／減碼",
    "PE": "本益比",
    "PB": "股價淨值比",
    "Provider has no institutional-flow data": "資料來源未提供法人／籌碼資料",
    "Not enough recent institutional-flow observations": "近期籌碼資料不足",
    "Recent institutional net-buying trend": "近期法人買賣超趨勢",
    "Price/volume history unavailable": "無價格或成交量資料",
    "Not enough price/volume history": "價格或成交量歷史不足",
    "Uptrend with expanding volume": "放量上升趨勢",
    "Downtrend with expanding volume": "放量下跌趨勢",
    "Mixed/neutral flow proxy": "資金流訊號中性",
    "N/A": "無法計算",
}

#格式輸出調整
def format_report(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 先將所有數值欄位保留到小數點後 3 位
    numeric_cols = df.select_dtypes(include=["number"]).columns
    df[numeric_cols] = df[numeric_cols].round(3)

    # 再把評分欄位改成整數
    score_cols = [
        "Valuation Percentile",
        "Growth Score",
        "Quality Score",
        "Buffett Score",
        "Fundamental Score",
        "RSI",
    ]

    for col in score_cols:
        if col in df.columns:
            df[col] = df[col].round(0).astype("Int64")

    return df

def analyze_universe(
    provider: DataProvider,
    tickers: list[str],
    config: dict[str, Any],
    language: str = "en",
) -> pd.DataFrame:
    valuation_engine = ValuationEngine(config)
    quality_engine = QualityEngine()
    score_engine = ScoringEngine(config)
    checklist = BuffettChecklist(config)
    decision_engine = DecisionEngine(config)
    profiles = ProfileResolver(config)
    targets = PriceTargetEngine()
    technical_engine = TechnicalAnalysisEngine(config)  
    chip_engine = ChipAnalysisEngine(config)
    us_flow_engine = USFlowProxyEngine(config)

    records: list[dict[str, Any]] = []

    for ticker in tickers:
        try:
            snapshot = provider.get_snapshot(ticker)
            prices = provider.get_price_history(ticker)
            historical_fundamentals = provider.get_historical_fundamentals(ticker)

            valuation = valuation_engine.calculate(snapshot, prices, historical_fundamentals)
            quality = quality_engine.calculate(snapshot)

            quality_score = score_engine.quality_score(quality, valuation.fcf_yield)
            valuation_score = score_engine.valuation_score(valuation.pe_percentile_5y)
            growth_score = score_engine.growth_score(quality)
            short_growth_score = score_engine.short_growth_score(quality) #額外新增短期動能評估
            short_growth_direction = score_engine.short_growth_direction(quality) #額外新增短期動能評價

            checks = checklist.evaluate(quality, snapshot)
            buffett_score = checklist.score(checks)

            total_score = score_engine.total_score(
                valuation_score,
                quality_score,
                growth_score,
                buffett_score,
            )

            classification = profiles.classify(ticker, snapshot.asset_type)
            rules = profiles.rules(classification, ticker)

            target = targets.calculate(snapshot, rules, valuation)
            technical = technical_engine.analyze(prices)     
            #技術面評價分數
            technical_score = score_engine.technical_score(
                technical,
                snapshot.current_price
            )
                
            if classification.market == "TW":

                chip = chip_engine.analyze(
                    provider.get_chip_data(ticker),
                    snapshot.current_price
                )

                flow_signal = chip.signal


                # 新增：如果未來 ChipAnalysisEngine 有提供，就直接使用
                volume_ratio = getattr(
                    chip,
                    "volume_ratio",
                    None
                )


                return_20d = getattr(
                    chip,
                    "return_20d",
                    None
                )


                stop_loss = chip.stop_loss
                take_profit = chip.take_profit

                flow_note = chip.reason


            else:

                us_flow = us_flow_engine.analyze(prices)

                flow_signal = us_flow.signal

                volume_ratio = us_flow.volume_ratio

                return_20d = us_flow.return_20d


                stop_loss = None
                take_profit = None

                flow_note = us_flow.reason
            #籌碼面評價分數
            flow_score = score_engine.flow_score(
                flow_signal,
                volume_ratio,
                return_20d
            )
            decision = decision_engine.decide(total_score)
                
            records.append({
                "Ticker": ticker,
                "Source": snapshot.source,
                "Market": classification.market,
                "Sector": classification.sector,
                "Price": snapshot.current_price,                
                "PE": valuation.pe,
                # 已公告最近四季 EPS
                "TTM EPS": snapshot.ttm_eps if snapshot.ttm_eps is not None else "N/A",
                # FinMind 月營收即時推估 EPS
                "EPS TTM Nowcast": snapshot.eps_ttm_nowcast if snapshot.eps_ttm_nowcast is not None else "N/A",
                "PE TTM Nowcast": snapshot.pe_ttm_nowcast if snapshot.pe_ttm_nowcast is not None else "N/A",
                # Yahoo / 市場共識預估
                "Forward EPS": snapshot.forward_eps if snapshot.forward_eps is not None else "N/A",
                "Forward PE": valuation.forward_pe if valuation.forward_pe is not None else "N/A",
                "PB": valuation.pb,
                "ROIC": quality.roic,
                "ROE": quality.roe,
                "FCF Yield": valuation.fcf_yield,
                "Revenue Growth 3M": quality.revenue_growth_3m,
                "Revenue Growth 6M": quality.revenue_growth_6m,
                "EPS Growth 3M": quality.eps_growth_3m,
                "EPS Growth 6M": quality.eps_growth_6m,
                "EPS CAGR 3Y": quality.eps_cagr_3y,
                "Short Growth Score": short_growth_score,  #額外新增
                "EPS Momentum Trend": short_growth_direction["eps_direction"], #額外評價
                "Revenue Momentum Trend": short_growth_direction["revenue_direction"], #額外評價
                "Short Growth Trend": short_growth_direction["overall_direction"], #額外評價
                "Valuation Percentile": valuation.pe_percentile_5y,
                #"Valuation Sample Count": valuation.pe_sample_count,
                #"Valuation History Years": valuation.pe_history_years,
                "Growth Score": growth_score,
                "Quality Score": quality_score,
                "Buffett Score": buffett_score,
                "Fundamental Score": total_score,
                "Recommendation": decision.label,
                "Stars": decision.stars,
                "Model Buy Price": target.model_buy_price,
                "Model Fair Price": target.model_fair_price,
                "Model Sell Price": target.model_sell_price,
                "Historical P05 Price": target.historical_P05_price,
                "Historical Buy Price": target.historical_buy_price,
                "Historical Fair Price": target.historical_fair_price,
                "Historical Sell Price": target.historical_sell_price,
                "Historical P95 Price": target.historical_P95_price,
                "Nowcast PE Percentile": valuation.nowcast_pe_percentile if valuation.nowcast_pe_percentile is not None else "N/A",
                "Fair Value Upside": target.upside_to_fair,
                "Target Metric": target.primary_metric,
                #"Valuation Basis": target.basis,
                "Technical Signal": technical.signal,
                "Technical Score": technical_score,
                "RSI": technical.rsi,
                "Support": technical.support,
                "Resistance": technical.resistance,
                "Flow Signal": flow_signal,
                "Flow Score": flow_score,
                "Flow Note": flow_note,
            })
        except Exception as exc:
            LOGGER.exception("Could not analyze %s", ticker)
            records.append({
                "Ticker": ticker,
                "Source": None,
                "Recommendation": "Data Error",
                "Error": str(exc),
            })

    report = pd.DataFrame(records)
    report = format_report(report)

    if language.lower() in {"zh", "zh-tw", "zh_tw"}:
        return report.replace(ZH_TW_VALUES).rename(columns=ZH_TW_COLUMNS)

    return report


#檔案日期改寫
def save_report(report, config, prefix=None):

    from datetime import datetime
    from pathlib import Path

    now = datetime.now()

    # 取得 config 名稱
    config_name = config.get(
        "_config_name",
        "default"
    )


    output_dir = (
        Path("reports")
        / config_name
        / now.strftime("%Y")
        / now.strftime("%m")
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    filename = (
        output_dir
        /
        f"{config_name}_Report_{now.strftime('%Y%m%d_%H%M%S')}.xlsx"
    )


    report.to_excel(
        filename,
        index=False
    )


    print(
        f"✅ Saved to: {filename.resolve()}"
    )

    return filename

In [49]:
#------------------------------------------------------------------
#Cell 1 — 初始化
#------------------------------------------------------------------
from pathlib import Path
import pandas as pd

# 顯示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# 載入設定與 provider / engines
config = load_config("config/US.yaml")    #Allstar 、 AI 、Fin 、TW 、US
provider = HybridTaiwanProvider(config)

valuation_engine = ValuationEngine(config)
quality_engine = QualityEngine()
score_engine = ScoringEngine(config)
checklist = BuffettChecklist(config)
decision_engine = DecisionEngine(config)
profiles = ProfileResolver(config)
targets = PriceTargetEngine()
technical_engine = TechnicalAnalysisEngine(config)
chip_engine = ChipAnalysisEngine(config)
us_flow_proxy_engine = USFlowProxyEngine(config)

print("Initialized.")

Initialized.


In [50]:
#------------------------------------------------------------------
#Cell 8 — 批量分析 / 匯出 Excel
#------------------------------------------------------------------
tickers = config["universe"]["taiwan"] + config["universe"]["us"]

report = analyze_universe(provider, tickers, config, language="zh-tw")
save_report(report, config)
report

✅ Saved to: C:\Users\wenhao.tsai\Project\WHID\Stock_Valuation\reports\US\2026\08\US_Report_20260828_103410.xlsx


,代號,資料來源,市場,產業,現價,本益比,近年EPS,即時EPS,即時本益比,預估EPS,預估本益比,股價淨值比,投入資本報酬率,股東權益報酬率,自由現金流殖利率,營收近三月成長率,營收近六月成長率,EPS 近三月成長率,EPS 近六月成長率,EPS 三年複合成長率,短期成長動能,EPS短期趨勢,營收短期趨勢,短期動能趨勢,歷史估值百分位,成長分數,品質分數,巴菲特檢核分數,基本面分數,投資建議,星等,模型預估便宜價,模型預估合理價,模型預估昂貴價,歷史估值05%價格,歷史估值25%價格,歷史估值50%價格,歷史估值75%價格,歷史估值95%價格,即時歷史估值位階,合理價上行空間,目標價依據,技術面訊號,技術面分數,相對強弱指標 RSI,支撐價,壓力價,資金流訊號,資金流分數,資金流說明
0,GOOG,Yahoo Finance (yfinance),US,科技,337.71,16.962,19.910,無法計算,無法計算,14.828,22.776,6.635,0.352,0.318,0.005,0.090,0.052,0.783,2.230,0.333,72.8,正向無趨勢,正向無趨勢,正向無趨勢,2,84,88,100,89,強力買進,★★★★★,497.818,568.338,596.112,391.925,497.818,568.338,596.112,761.728,無法計算,0.683,本益比,賣出／減碼訊號,65.0,43,318.34,375.35,持有,35.0,"Mixed/neutral flow proxy (vol_ratio=0.74, return_20d=1.21%)"
1,META,Yahoo Finance (yfinance),US,科技,571.10,21.510,26.550,無法計算,無法計算,34.835,16.394,5.571,0.278,0.278,0.015,0.080,0.015,-0.408,-0.304,0.398,26.0,正向轉強負向,正向無趨勢,正向但營收／EPS趨勢不同,9,80,74,100,81,強力買進,★★★★★,698.780,796.932,917.383,546.380,698.780,796.932,917.383,1082.394,無法計算,0.395,本益比,賣出／減碼訊號,70.0,47,539.03,681.31,持有,45.0,"Mixed/neutral flow proxy (vol_ratio=0.91, return_20d=5.95%)"
2,MSFT,Yahoo Finance (yfinance),US,科技,505.06,28.121,17.960,無法計算,無法計算,23.573,21.425,8.479,0.251,0.302,0.004,0.086,0.107,0.126,-0.068,0.229,40.2,正向波動,正向但減速,正向但營收／EPS趨勢不同,14,76,84,100,83,強力買進,★★★★★,583.423,638.292,700.126,484.485,583.423,638.292,700.126,783.483,無法計算,0.264,本益比,賣出／減碼訊號,15.0,70,352.83,506.06,持有,55.0,"Mixed/neutral flow proxy (vol_ratio=0.57, return_20d=11.96%)"
3,NVDA,Yahoo Finance (yfinance),US,科技,227.98,34.913,6.530,無法計算,無法計算,14.954,15.246,28.250,0.892,0.763,0.008,0.198,0.432,0.358,0.838,2.042,94.9,正向但減速,正向但減速,正向但減速,7,100,92,100,96,強力買進,★★★★★,384.511,611.588,1030.330,228.202,384.511,611.588,1030.330,1837.069,無法計算,1.683,本益比,持有／等待,20.0,61,190.01,227.98,持有,70.0,"Mixed/neutral flow proxy (vol_ratio=1.21, return_20d=16.89%)"
4,NFLX,Yahoo Finance (yfinance),US,科技,79.84,25.131,3.177,無法計算,無法計算,3.821,20.896,11.026,0.312,0.413,0.076,0.025,0.042,-0.350,0.429,0.365,41.4,正向轉負向,正向但減速,正向但營收／EPS趨勢不同,7,99,84,100,92,強力買進,★★★★★,121.549,153.150,179.536,80.683,121.549,153.150,179.536,227.971,無法計算,0.918,本益比,持有／等待,38.0,58,67.60,82.64,持有,45.0,"Mixed/neutral flow proxy (vol_ratio=0.58, return_20d=9.12%)"
5,BABA,Yahoo Finance (yfinance),US,科技,116.31,2.702,43.040,無法計算,無法計算,9.326,12.471,1.740,0.051,0.098,-0.286,-0.146,-0.018,0.757,0.193,0.170,37.8,負向轉強正向,負向強惡化,負向但營收／EPS趨勢不同,38,49,52,50,52,持有,★★★☆☆,113.262,122.538,136.045,93.928,113.262,122.538,136.045,164.723,無法計算,0.054,本益比,持有／等待,55.0,42,94.81,132.32,持有,55.0,"Mixed/neutral flow proxy (vol_ratio=1.55, return_20d=-0.01%)"
6,ADBE,Yahoo Finance (yfinance),US,科技,289.15,16.542,17.480,無法計算,無法計算,27.490,10.518,10.017,0.533,0.613,0.080,0.034,0.068,-0.076,-0.045,0.182,9.5,正向轉強負向,正向但減速,正向但營收／EPS趨勢不同,16,84,80,100,84,強力買進,★★★★★,495.093,645.276,813.803,251.058,495.093,645.276,813.803,961.770,無法計算,1.232,本益比,持有／等待,20.0,68,193.41,289.15,持有,55.0,"Mixed/neutral flow proxy (vol_ratio=0.67, return_20d=16.64%)"
7,V,Yahoo Finance (yfinance),US,金融,379.66,32.284,11.760,無法計算,無法計算,15.004,25.303,20.120,0.443,0.529,0.029,0.036,0.067,-0.054,-0.020,0.134,14.7,正向轉強負向,正向但減速,正向但營收／EPS趨勢不同,35,69,66,100,70,買進,★★★★☆,373.015,386.353,409.270,349.269,373.015,386.353,409.270,427.698,無法計算,0.018,本益比,持有／等待,20.0,64,312.40,384.14,持有,35.0,"Mixed/neutral flow proxy (vol_ratio=0.87, return_20d=3.66%)"
8,AMZN,Yahoo Finance (yfinance),US,科技,256.26,20.616,12.430,無法計算,無法計算,10.391,24.663,5.009,0.119,0.189,0.001,-0.149,0.007,1.068,1.949,NaN,63.1,正向無趨勢,正向轉負向,正向但營收／EPS趨勢不同,0,73,58,80,72,買進,★★★★☆,434.029,512.177,756.081,354.745,434.029,512.177,756.081,900.185,無法計算,0.999,本益比,持有／等待,55.0,47,226.65,284.02,持有,45.0,"Mixed/neutral flow proxy (vol_ratio=0.60, return_20d=8.82%)"
9,ORCL,Yahoo Finance (yfinance),US,科技,151.94,26.062,5.830,無法計算,無法計算,10.926,13.906,11.652,0.101,0.402,-0.056,0.116,0.195,0.142,-0.310,0.238,51.9,正向波動,正向但減速,正向但營收／EPS趨勢不同,3,91,64,50,77,買進,★★★★☆,207.682,216.932,259.931,184.345,207.682,216.932,259.931,335.110,無法計算,0.428,本益比,賣出／減碼訊號,40.0,57,114.99,236.34,持有,55.0,"Mixe

In [53]:
# ------------------------------------------------------------------
# Cell — Historical Stock Report Collector (Enhanced)
# ------------------------------------------------------------------
from pathlib import Path
from datetime import datetime
import pandas as pd

report_dir = Path("reports")

# 只抓符合命名規則的歷史報表
files = sorted(report_dir.rglob("US_Report_*.xlsx"))       #US AI FIN

all_reports = []

for file in files:
    try:
        timestamp_str = file.stem.split("_Report_")[1]
        snapshot_time = datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

        df = pd.read_excel(file)

        parents = file.parents
        config_name = parents[2].name if len(parents) >= 3 else "unknown"

        df.insert(0, "SnapshotTime", snapshot_time)
        df.insert(1, "Config", config_name)
        df.insert(2, "SourceFile", file.name)

        all_reports.append(df)
        print(f"Loaded: {file.name}")

    except Exception as e:
        print(f"Skip {file.name}: {e}")

# -------------------------------------------------------------
# 空資料保護
# -------------------------------------------------------------
if not all_reports:
    raise ValueError("No report files loaded. Please check the reports folder or file naming pattern.")

# -------------------------------------------------------------
# 合併全部歷史報表
# -------------------------------------------------------------
history_df = pd.concat(all_reports, ignore_index=True)
history_df = history_df.sort_values("SnapshotTime").reset_index(drop=True)

print(f"\nTotal rows: {len(history_df)}")
if "代號" in history_df.columns:
    print(f"Total tickers: {history_df['代號'].nunique()}")

# -------------------------------------------------------------
# 匯出資料夾
# -------------------------------------------------------------
history_dir = report_dir / "history"
history_dir.mkdir(parents=True, exist_ok=True)

output_file = history_dir / f"WHID_History_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

# -------------------------------------------------------------
# 最新摘要
# -------------------------------------------------------------
if "代號" in history_df.columns:
    latest_df = (
        history_df
        .sort_values("SnapshotTime")
        .groupby("代號", as_index=False)
        .tail(1)
        .sort_values(["Config", "基本面分數"], ascending=[True, False], na_position="last")
        .reset_index(drop=True)
    )
else:
    latest_df = history_df.copy()

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # 1. 全部資料總表
    history_df.to_excel(writer, sheet_name="AllReports", index=False)

    # 2. 最新摘要
    latest_df.to_excel(writer, sheet_name="LatestSummary", index=False)

    # 3. 各股票分頁
    if "代號" in history_df.columns:
        tickers = history_df["代號"].dropna().unique()
        used_sheet_names = set(["AllReports", "LatestSummary"])

        for ticker in tickers:
            stock_df = history_df[history_df["代號"] == ticker].copy()
            stock_df = stock_df.sort_values("SnapshotTime").reset_index(drop=True)

            # -------------------------------------------------
            # 新增：現價溢價比例 = 現價 / 基本面買點
            # 新增：現價溢價比例變化
            # -------------------------------------------------
            if "現價" in stock_df.columns and "基本面買點" in stock_df.columns:
                stock_df["現價溢價比例"] = stock_df["現價"] / stock_df["基本面買點"]
                stock_df["現價溢價比例變化"] = stock_df["現價溢價比例"].diff()

            # -------------------------------------------------
            # 其他變化欄位
            # -------------------------------------------------
            if "現價" in stock_df.columns:
                stock_df["現價變化"] = stock_df["現價"].diff()

            if "綜合分數" in stock_df.columns:
                stock_df["基本面分數變化"] = stock_df["基本面分數"].diff()

            if "歷史估值百分位" in stock_df.columns:
                stock_df["歷史估值百分位變化"] = stock_df["歷史估值百分位"].diff()

            if "投資建議" in stock_df.columns:
                stock_df["投資建議變化"] = stock_df["投資建議"].shift(1).fillna("") + " → " + stock_df["投資建議"]
                stock_df.loc[stock_df.index == 0, "投資建議變化"] = stock_df.loc[stock_df.index == 0, "投資建議"]

            # sheet name（代號即可，避免中文名稱管理）
            base_name = str(ticker)[:31]
            sheet_name = base_name
            counter = 1

            while sheet_name in used_sheet_names:
                suffix = f"_{counter}"
                sheet_name = base_name[:31 - len(suffix)] + suffix
                counter += 1

            used_sheet_names.add(sheet_name)

            stock_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved: {output_file}")

Loaded: US_Report_20260804_143307.xlsx
Loaded: US_Report_20260805_135620.xlsx
Loaded: US_Report_20260806_143936.xlsx
Loaded: US_Report_20260807_185103.xlsx
Loaded: US_Report_20260810_140856.xlsx
Loaded: US_Report_20260810_155614.xlsx
Loaded: US_Report_20260811_080144.xlsx
Loaded: US_Report_20260811_144136.xlsx
Loaded: US_Report_20260811_151035.xlsx
Loaded: US_Report_20260812_140537.xlsx
Loaded: US_Report_20260813_151643.xlsx
Loaded: US_Report_20260814_083129.xlsx
Loaded: US_Report_20260814_160946.xlsx
Loaded: US_Report_20260815_093239.xlsx
Loaded: US_Report_20260817_165135.xlsx
Loaded: US_Report_20260818_075700.xlsx
Loaded: US_Report_20260818_151146.xlsx
Loaded: US_Report_20260819_070946.xlsx
Loaded: US_Report_20260819_152207.xlsx
Loaded: US_Report_20260820_133944.xlsx
Loaded: US_Report_20260821_085333.xlsx
Loaded: US_Report_20260821_134037.xlsx
Loaded: US_Report_20260823_231523.xlsx
Loaded: US_Report_20260825_153548.xlsx
Loaded: US_Report_20260826_142615.xlsx
Loaded: US_Report_2026082